# V15ADage — the merged benchmark with age taken out, two ways

V7ADmerged showed that one training cohort, **GSE33000**, is confounded by age: its
controls are 14 years younger than its AD donors, and age alone separates them with
AUC 0.86 inside that cohort. This notebook re-runs the **identical** V7 pipeline
(same models, hyperparameters, 20 real + 10 label-permuted donor-grouped splits,
permutation FDR, stability selection, g:Profiler enrichment) twice:

**Arm A — age-corrected expression (primary).** In every split, for every gene and
every cohort, the slope of expression on age is estimated on the *training-partition
control arrays only* and the age-predicted part is subtracted from all arrays of that
cohort: `x' = x − b·(age − mean control age)`. Validation and test donors never
contribute to the slopes, and the slopes come from controls so that the disease
effect, which is older on average, is not removed with them. The label-permuted null
runs go through exactly the same correction.

**Arm B — age-matched donors (sensitivity).** Within each cohort, donors are
frequency-matched on age in 5-year bands, keeping the cohort's AD:control ratio
(seed 0); bands holding only one diagnosis are dropped. Nothing is modelled; age
alone is at chance (AUC ≈ 0.5) in every cohort afterwards. The unchanged pipeline
then runs on the matched donors.

**Part C** sets both arms against V7: headline and per-cohort AUC (with what age
alone achieves in each cohort), the gene lists and how many V7 genes survive, the 18
genes V7's two models shared, and whether the selected genes are age genes.

Inputs: `ad-merged-geo-cohorts` (the matrix) and `v7admerged` (the reference run).
Outputs: `A_age_corrected/` and `B_age_matched/` (each the full V7 file set) and the
comparison files plus `V15_REPORT.txt` at the top level.


## Setup — data, models, attribution, split trainer


In [ ]:
# ======================================================================
# Cell 1 — configuration and data
# ======================================================================
import os, json, warnings, numpy as np, pandas as pd, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix,
                             silhouette_score, adjusted_rand_score,
                             davies_bouldin_score, calinski_harabasz_score)
from scipy import stats
from scipy.stats import rankdata, mannwhitneyu, wilcoxon, ttest_rel

warnings.filterwarnings("ignore")

N_REAL_RUNS = 20          # independent retrained splits, true labels
N_NULL_RUNS = 10          # independent retrained splits, permuted labels
ALPHA       = 0.05        # FDR threshold
IG_STEPS    = 50          # Integrated Gradients path steps
LATENT_DIM  = 8
HIDDEN_DIM  = 256
BETA        = 2.0
GAMMA       = 10.0
MAX_EPOCHS  = 200
PATIENCE    = 20
KL_ANNEAL   = 50
REPRESENTATIVE_SEED = 19  # the split used for single-run illustrations
AGE_MIN     = 60          # V6: age-matched cohort, donors aged >= AGE_MIN only
AGE_RESID   = False       # V15: set per arm by set_arm()

# Stability-selection windows. The significance test (Mann-Whitney against
# the permuted null) is identical for both methods; only the top-K window is
# widened for the CVAE, whose per-run rankings carry extra estimator variance
# (weight init, batch order, dropout, early-stopping epoch) on top of the
# split variance PLS-DA also sees.
METHOD_PARAMS = {
    "PLS-DA_VIP": {"top_k": 100, "stability_threshold": 0.50},
    "CVAE_IG":    {"top_k": 175, "stability_threshold": 0.40},
}

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

# The published numbers were produced by a session in which CUDA was present but
# its kernels were unusable, so the whole benchmark executed on CPU. PLS-DA is a
# deterministic decomposition and reproduces anywhere; the CVAE does not, because
# dropout masks are drawn from whichever device generator is active. Pinning the
# execution path to CPU is therefore what makes this notebook a reproduction of
# the manuscript rather than a fresh experiment. Set to False to run on GPU --
# expect the CVAE gene list to move, exactly as the paper's Section 4.7 measures.
FORCE_CPU = True


def pick_device():
    """torch.cuda.is_available() is not sufficient on Kaggle: the session can
    be handed a GPU whose compute capability the installed torch build has no
    kernels for, which only surfaces as `no kernel image is available for
    execution on the device` once a real kernel launches. Run a tiny matmul
    and fall back to CPU if it raises."""
    if FORCE_CPU:
        print("FORCE_CPU set; running on CPU as the published run did")
        return torch.device("cpu")
    if torch.cuda.is_available():
        try:
            a = torch.randn(64, 64, device="cuda")
            torch.mm(a, a).sum().item()
            torch.nn.Linear(64, 64).cuda()(a).sum().item()
            return torch.device("cuda")
        except Exception as e:                       # noqa: BLE001
            print(f"CUDA present but unusable ({type(e).__name__}: {e}); using CPU")
    return torch.device("cpu")


device = pick_device()
if device.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)
print("device:", device, "| torch", torch.__version__)

# Record the environment rather than leaving it to be inferred from a log. A
# CVAE result is only reproducible against a stated environment, so the versions
# that produced these numbers are part of the output, not part of the metadata.
import platform, sys, importlib


def _ver(mod):
    try:
        return getattr(importlib.import_module(mod), "__version__", "unknown")
    except Exception:                                # noqa: BLE001
        return "not installed"


env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "scikit_learn": _ver("sklearn"), "scipy": _ver("scipy"),
       "device": str(device), "force_cpu": FORCE_CPU,
       "cuda_available": torch.cuda.is_available(),
       "cpu_count": os.cpu_count(), "torch_num_threads": torch.get_num_threads()}
pd.DataFrame([{"key": k, "value": v} for k, v in env.items()]).to_csv(
    f"{OUT}/env_report.csv", index=False)
for k, v in env.items():
    print(f"  {k:16} {v}")

import glob
_hits = glob.glob('/kaggle/input/**/merged_expression.csv.gz', recursive=True)
assert _hits, "attach the ad-merged-geo-cohorts notebook output as an input"
MERGED = os.path.dirname(_hits[0])
print("merged data from", MERGED)

df_genes = pd.read_csv(f"{MERGED}/merged_expression.csv.gz")
df_labels = pd.read_csv(f"{MERGED}/merged_labels.csv").rename(columns={'GSM_ID': 'Subject_ID'})
df_meta = pd.read_csv(f"{MERGED}/merged_metadata.csv")
df = (df_genes.merge(df_labels, on='Subject_ID', how='inner', validate='one_to_one')
              .merge(df_meta, on='Subject_ID', how='inner', validate='one_to_one'))
assert len(df) == len(df_genes) == len(df_labels)
assert (df['GROUP'] == df['dx']).all()
print(pd.read_csv(f"{MERGED}/cohort_table.csv").to_string(index=False))
df = df[df['age'] >= AGE_MIN].reset_index(drop=True)      # already applied upstream
print(f"age >= {AGE_MIN}: {len(df)} arrays from {df.donor.nunique()} donors in "
      f"{df.cohort.nunique()} cohorts")
print(df.groupby(['cohort', 'GROUP']).agg(arrays=('Subject_ID', 'size'),
                                          donors=('donor', 'nunique')).to_string())

meta_cols   = ['Subject_ID', 'GROUP', 'donor', 'region', 'age', 'sex', 'dx', 'cohort', 'platform']
gene_cols   = [c for c in df.columns if c not in meta_cols]
donor_ids   = df['donor'].values
regions     = df['region'].values
ages        = df['age'].values.astype(float)
cohorts     = df['cohort'].values
subject_ids = df['Subject_ID'].values
X_raw       = df[gene_cols].values.astype(np.float32)
label_encoder = LabelEncoder()
y_raw       = label_encoder.fit_transform(df['GROUP'].values)
num_classes = len(label_encoder.classes_)
n_genes     = len(gene_cols)
AD_IDX      = int(np.where(label_encoder.classes_ == 'AD')[0][0])

print(f"arrays={X_raw.shape[0]}  donors={len(np.unique(donor_ids))}  genes={n_genes}  "
      f"classes={list(label_encoder.classes_)}")
print(pd.Series(df['GROUP']).value_counts().to_string())

pd.DataFrame({
    "n_subjects": [X_raw.shape[0]], "n_genes": [n_genes],
    "n_AD": [int((y_raw == AD_IDX).sum())],
    "n_control": [int((y_raw != AD_IDX).sum())],
    "class_order": ["|".join(label_encoder.classes_)],
    "AD_index": [AD_IDX],
    "n_donors": [len(np.unique(donor_ids))],
    "n_donors_AD": [len(np.unique(donor_ids[y_raw == AD_IDX]))],
    "n_donors_control": [len(np.unique(donor_ids[y_raw != AD_IDX]))],
    "age_min": [AGE_MIN],
    "n_cohorts": [len(np.unique(cohorts))],
    "age_median_AD": [float(np.median(df.loc[y_raw == AD_IDX].drop_duplicates('donor').age))],
    "age_median_control": [float(np.median(df.loc[y_raw != AD_IDX].drop_duplicates('donor').age))],
}).to_csv(f"{OUT}/cohort_summary.csv", index=False)
df[['Subject_ID', 'GROUP', 'cohort', 'platform', 'donor', 'region', 'age', 'sex']].to_csv(
    f"{OUT}/sample_metadata.csv", index=False)

# ======================================================================
# V15 — the two ways of taking age out
# ======================================================================
X_ALL, Y_ALL, DF_ALL = X_raw.copy(), y_raw.copy(), df.copy()
ROOT_OUT = OUT


def age_residualise(X, fit_rows, y_use):
    """Remove normal ageing from every gene, cohort by cohort. The age slope of
    each gene is estimated on the CONTROL arrays among `fit_rows` (the training
    partition of a split) and subtracted from every array of that cohort:
    x' = x - b * (age - mean age of those controls). Controls only, because in
    the confounded cohort AD donors are older and a slope fitted on everyone
    would carry part of the disease effect away with the age effect."""
    Xr = X.copy()
    for c in np.unique(cohorts):
        rows = np.where(cohorts == c)[0]
        fit = fit_rows[(cohorts[fit_rows] == c) & (y_use[fit_rows] != AD_IDX)]
        if len(fit) < 8 or np.ptp(ages[fit]) == 0:
            continue
        a = ages[fit]; a0 = a.mean()
        xc = X[fit] - X[fit].mean(0)
        slope = ((a - a0)[:, None] * xc).sum(0) / ((a - a0) ** 2).sum()
        Xr[rows] = X[rows] - np.outer(ages[rows] - a0, slope).astype(X.dtype)
    return Xr


def age_match_mask(seed=0, width=5):
    """Frequency-match donors on age within each cohort: inside every 5-year
    band keep AD and control donors in the cohort's overall AD:control ratio,
    subsampling whichever group is in excess; drop bands with one group only."""
    rs = np.random.RandomState(seed)
    dn = DF_ALL.drop_duplicates("donor")[["donor", "cohort", "GROUP", "age"]]
    keep = []
    for c, d in dn.groupby("cohort"):
        r = (d.GROUP == "AD").sum() / (d.GROUP == "Control").sum()
        for _, h in d.groupby((d.age // width) * width):
            a = np.sort(h[h.GROUP == "AD"].donor.values); k = np.sort(h[h.GROUP == "Control"].donor.values)
            if len(a) == 0 or len(k) == 0:
                continue
            if len(a) >= r * len(k):
                na, nk = max(1, int(round(r * len(k)))), len(k)
            else:
                na, nk = len(a), max(1, int(round(len(a) / r)))
            keep += list(rs.choice(a, min(na, len(a)), replace=False))
            keep += list(rs.choice(k, min(nk, len(k)), replace=False))
    return DF_ALL.donor.isin(set(keep)).values


def set_arm(name, keep, resid):
    """Point every global the V7 cells read at one arm's data and output folder."""
    global df, X_raw, y_raw, donor_ids, regions, ages, cohorts, subject_ids, OUT, AGE_RESID
    df = DF_ALL[keep].reset_index(drop=True)
    X_raw, y_raw = X_ALL[keep], Y_ALL[keep]
    donor_ids, regions = df.donor.values, df.region.values
    ages, cohorts, subject_ids = df.age.values.astype(float), df.cohort.values, df.Subject_ID.values
    AGE_RESID = resid
    OUT = f"{ROOT_OUT}/{name}"
    os.makedirs(OUT, exist_ok=True)
    pd.DataFrame({
        "n_subjects": [X_raw.shape[0]], "n_genes": [n_genes],
        "n_AD": [int((y_raw == AD_IDX).sum())], "n_control": [int((y_raw != AD_IDX).sum())],
        "class_order": ["|".join(label_encoder.classes_)], "AD_index": [AD_IDX],
        "n_donors": [len(np.unique(donor_ids))],
        "n_donors_AD": [len(np.unique(donor_ids[y_raw == AD_IDX]))],
        "n_donors_control": [len(np.unique(donor_ids[y_raw != AD_IDX]))],
        "age_min": [AGE_MIN], "n_cohorts": [len(np.unique(cohorts))],
        "age_median_AD": [float(np.median(df.loc[y_raw == AD_IDX].drop_duplicates('donor').age))],
        "age_median_control": [float(np.median(df.loc[y_raw != AD_IDX].drop_duplicates('donor').age))],
    }).to_csv(f"{OUT}/cohort_summary.csv", index=False)
    df[['Subject_ID', 'GROUP', 'cohort', 'platform', 'donor', 'region', 'age', 'sex']].to_csv(
        f"{OUT}/sample_metadata.csv", index=False)
    rows = []
    for c, d in df.drop_duplicates("donor").groupby("cohort"):
        yy = (d.GROUP == "AD").astype(int).values
        rows.append({"cohort": c, "AD_donors": int(yy.sum()), "control_donors": int((1 - yy).sum()),
                     "arrays": int((df.cohort == c).sum()),
                     "age_median_AD": d[d.GROUP == "AD"].age.median(),
                     "age_median_control": d[d.GROUP == "Control"].age.median(),
                     "auc_age_alone": roc_auc_score(yy, d.age.values) if 0 < yy.sum() < len(yy) else np.nan})
    bal = pd.DataFrame(rows)
    bal.to_csv(f"{OUT}/arm_age_balance.csv", index=False)
    print(f"ARM {name}: {len(df)} arrays, {len(np.unique(donor_ids))} donors, "
          f"age correction {'ON' if resid else 'off'}")
    print(bal.round(3).to_string(index=False))

print("V15 age tools ready")


In [ ]:
# ======================================================================
# Cell 2 — model, losses and shared helpers
# ======================================================================
class GeneExpressionDataset(Dataset):
    def __init__(self, X, y, num_classes):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y).long()
        self.num_classes = num_classes
    def __len__(self):  return self.X.shape[0]
    def __getitem__(self, i):
        y = self.y[i]
        cond = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()
        return self.X[i], cond, y


class CVAE(nn.Module):
    def __init__(self, input_dim, cond_dim, latent_dim=8, hidden_dim=256, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + cond_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2))
        self.mu_layer     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim))
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes))

    def encode(self, x, cond):
        h = self.encoder(torch.cat([x, cond], dim=1))
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def decode(self, z, cond):
        return self.decoder(torch.cat([z, cond], dim=1))

    def forward(self, x, cond):
        mu, logvar = self.encode(x, cond)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, cond), mu, logvar, self.classifier(mu)


def cvae_loss(recon_x, x, mu, logvar, logits, y, beta, gamma):
    recon = nn.MSELoss(reduction='mean')(recon_x, x)
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    ce    = nn.CrossEntropyLoss()(logits, y)
    return recon + beta * kl + gamma * ce


def neutral_cond(n, num_classes, device):
    """Label-agnostic conditioning: uniform over classes, so the encoder is
    given no information about the subject's diagnosis."""
    return torch.full((n, num_classes), 1.0 / num_classes, device=device)


def encode_all(model, X, y, num_classes, device, mode="true", batch_size=256):
    """Return (mu, logvar) for every row of X. mode='true' conditions on the
    observed label; mode='neutral' conditions on the uniform vector."""
    model.eval()
    mus, lvs = [], []
    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32, device=device)
            if mode == "true":
                yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
                cb = torch.nn.functional.one_hot(yb, num_classes).float()
            else:
                cb = neutral_cond(len(xb), num_classes, device)
            mu, lv = model.encode(xb, cb)
            mus.append(mu.cpu().numpy()); lvs.append(lv.cpu().numpy())
    return np.vstack(mus), np.vstack(lvs)


def reconstruct_cvae(model, X, y, num_classes, device, mode="true", batch_size=256):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32, device=device)
            if mode == "true":
                yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
                cb = torch.nn.functional.one_hot(yb, num_classes).float()
            else:
                cb = neutral_cond(len(xb), num_classes, device)
            mu, _ = model.encode(xb, cb)
            out.append(model.decode(mu, cb).cpu().numpy())   # decode the mean
    return np.vstack(out)


def per_sample_kl(mu, logvar):
    """KL(q(z|x,c) || N(0,I)) per subject, summed over latent dimensions (nats)."""
    return -0.5 * np.sum(1.0 + logvar - mu ** 2 - np.exp(logvar), axis=1)


def cluster_metrics(Z, y):
    km = KMeans(n_clusters=len(np.unique(y)), n_init=10, random_state=42)
    labels = km.fit_predict(Z)
    return {
        "silhouette":        silhouette_score(Z, y),
        "ari":               adjusted_rand_score(y, labels),
        # V6: scored against the diagnosis, like silhouette (v5 used the
        # k-means partition, which measures k-means rather than diagnosis)
        "davies_bouldin":    davies_bouldin_score(Z, y),
        "calinski_harabasz": calinski_harabasz_score(Z, y),
    }


def inter_intra_ratio(Z, y):
    """Mean between-class centroid distance over mean within-class spread."""
    cents = np.array([Z[y == c].mean(axis=0) for c in np.unique(y)])
    inter = np.linalg.norm(cents[0] - cents[1])
    intra = np.mean([np.linalg.norm(Z[y == c] - cents[i], axis=1).mean()
                     for i, c in enumerate(np.unique(y))])
    return inter / (intra + 1e-12)


def vip_scores(pls):
    """Variable Importance in Projection (Wold et al. 2001): a gene's weight
    aggregated over all components, weighted by the Y-variance each explains."""
    t, w, q = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    p, _ = w.shape
    s = np.sum(t ** 2, axis=0) * np.sum(q ** 2, axis=0)
    w_norm = w / (np.linalg.norm(w, axis=0, keepdims=True) + 1e-12)
    return np.sqrt(p * (w_norm ** 2) @ s / (np.sum(s) + 1e-12))


def bh_fdr(pvals):
    pvals = np.asarray(pvals); n = len(pvals)
    order = np.argsort(pvals)
    q = pvals[order] * n / np.arange(1, n + 1)
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0, 1)
    out = np.empty(n); out[order] = q
    return out

print("helpers defined")

In [ ]:
# ======================================================================
# Cell 3 — Integrated Gradients
# ======================================================================
!pip install captum umap-learn -q
from captum.attr import IntegratedGradients


def ig_attributions(model, X, y, num_classes, device, n_steps=IG_STEPS):
    """Attribute the true-class classifier logit back to genes, relative to a
    zero (= training-mean, post-standardisation) baseline."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.long, device=device)
    cond = torch.nn.functional.one_hot(y_t, num_classes).float()

    def forward_fn(x, cond_arg):
        mu, _ = model.encode(x, cond_arg)
        return model.classifier(mu)

    # cond must travel through additional_forward_args: captum expands
    # `inputs` to (batch * n_steps, ...) for the path integral and only
    # applies that expansion to tensors it is told about.
    # internal_batch_size caps that expansion — without it the path integral
    # materialises n_test * n_steps * n_genes floats in one allocation.
    attr = IntegratedGradients(forward_fn).attribute(
        X_t, torch.zeros_like(X_t), target=y_t,
        additional_forward_args=(cond,), n_steps=n_steps,
        internal_batch_size=64)
    return attr.detach().cpu().numpy()


def cvae_latent_saliency(model, X, y, num_classes, device, mode="true", batch_size=128):
    """|d mu_k / d x_j| averaged over subjects — the latent-to-gene map."""
    model.eval()
    accum = np.zeros((model.latent_dim, X.shape[1]), dtype=np.float64)
    n_seen = 0
    for s in range(0, len(X), batch_size):
        xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32,
                          device=device, requires_grad=True)
        if mode == "true":
            yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
            cb = torch.nn.functional.one_hot(yb, num_classes).float()
        else:
            cb = neutral_cond(len(xb), num_classes, device)
        mu, _ = model.encode(xb, cb)
        for k in range(model.latent_dim):
            g = torch.autograd.grad(mu[:, k].sum(), xb, retain_graph=(k < model.latent_dim - 1))[0]
            accum[k] += g.abs().sum(dim=0).detach().cpu().numpy()
        n_seen += len(xb)
    return accum / max(n_seen, 1)

print("attribution helpers ready")

In [ ]:
# ======================================================================
# Cell 4 — one split, trained end to end
# ======================================================================
def train_one_split(seed, X_raw, y_raw, permute_labels=False):
    """60/20/20 split over DONORS, stratified by diagnosis, scaler fit on train
    only, PLS-DA and CVAE trained on the identical partition. A donor's arrays
    (up to four brain regions) always land in the same partition."""
    donor_lab = pd.Series(y_raw).groupby(donor_ids).first()      # donor -> label
    donor_coh = pd.Series(cohorts).groupby(donor_ids).first()    # donor -> cohort
    if permute_labels:   # null: permute diagnoses across donors WITHIN each cohort
        rs = np.random.RandomState(seed + 999_000)
        perm = donor_lab.copy()
        for c in sorted(donor_coh.unique()):
            ids = donor_coh.index[donor_coh == c]
            perm.loc[ids] = rs.permutation(donor_lab.loc[ids].values)
        donor_lab = perm
    y_use = donor_lab.loc[donor_ids].values
    strata = donor_coh.astype(str) + "|" + donor_lab.astype(str)   # cohort x diagnosis

    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    d_all = donor_lab.index.values
    d_tr, d_tmp = train_test_split(d_all, test_size=0.4, stratify=strata.loc[d_all].values,
                                   random_state=seed)
    d_va, d_te  = train_test_split(d_tmp, test_size=0.5,
                                   stratify=strata.loc[d_tmp].values, random_state=seed)
    idx_tr = np.where(np.isin(donor_ids, d_tr))[0]
    idx_va = np.where(np.isin(donor_ids, d_va))[0]
    idx_te = np.where(np.isin(donor_ids, d_te))[0]
    assert not (set(donor_ids[idx_tr]) & set(donor_ids[idx_te])), "donor leak train/test"
    assert not (set(donor_ids[idx_tr]) & set(donor_ids[idx_va])), "donor leak train/val"
    assert not (set(donor_ids[idx_va]) & set(donor_ids[idx_te])), "donor leak val/test"
    assert len(idx_tr) + len(idx_va) + len(idx_te) == len(X_raw)

    y_tr, y_va, y_te = y_use[idx_tr], y_use[idx_va], y_use[idx_te]
    # V15: arm A removes normal ageing using the training controls of THIS split
    X_use = age_residualise(X_raw, idx_tr, y_use) if AGE_RESID else X_raw
    scaler  = StandardScaler().fit(X_use[idx_tr])
    X_tr = scaler.transform(X_use[idx_tr]).astype(np.float32)
    X_va = scaler.transform(X_use[idx_va]).astype(np.float32)
    X_te = scaler.transform(X_use[idx_te]).astype(np.float32)

    # ---- PLS-DA -------------------------------------------------------
    ohe = OneHotEncoder(sparse_output=False)
    plsda = PLSRegression(n_components=LATENT_DIM)
    plsda.fit(X_tr, ohe.fit_transform(y_tr.reshape(-1, 1)))

    # ---- CVAE ---------------------------------------------------------
    # V6: donor-grouped splits give a variable training size; if it leaves a
    # final batch of one array, BatchNorm cannot train on it, so drop only then.
    tr_loader = DataLoader(GeneExpressionDataset(X_tr, y_tr, num_classes), batch_size=32, shuffle=True,
                           drop_last=(len(X_tr) % 32 == 1))
    va_loader = DataLoader(GeneExpressionDataset(X_va, y_va, num_classes), batch_size=32, shuffle=False)

    model = CVAE(X_raw.shape[1], num_classes, LATENT_DIM, HIDDEN_DIM, num_classes).to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=10)

    best_acc, best_state, waited, stopped_at = 0.0, None, 0, MAX_EPOCHS
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        kl_w = min(1.0, epoch / KL_ANNEAL)
        for x, cond, yb in tr_loader:
            x, cond, yb = x.to(device), cond.to(device), yb.to(device)
            opt.zero_grad()
            recon, mu, logvar, logits = model(x, cond)
            cvae_loss(recon, x, mu, logvar, logits, yb, BETA * kl_w, GAMMA).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

        model.eval(); preds, true = [], []
        with torch.no_grad():
            for x, cond, yb in va_loader:
                _, _, _, logits = model(x.to(device), cond.to(device))
                preds.extend(torch.argmax(logits, 1).cpu().numpy()); true.extend(yb.numpy())
        val_acc = accuracy_score(true, preds)
        sched.step(val_acc)

        if val_acc > best_acc + 1e-3:
            best_acc, best_state, waited = val_acc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= PATIENCE:
                stopped_at = epoch; break

    if best_state is not None:
        model.load_state_dict(best_state)

    return dict(plsda=plsda, model=model, scaler=scaler,
                X_tr=X_tr, X_va=X_va, X_te=X_te, y_tr=y_tr, y_va=y_va, y_te=y_te,
                idx_tr=idx_tr, idx_va=idx_va, idx_te=idx_te,
                best_val_acc=best_acc, epochs_run=stopped_at)

print("split trainer ready")

## Part A — age-corrected expression (primary)

Every split removes each gene's normal-ageing slope, estimated per cohort on that split's training controls, before scaling and fitting.


In [ ]:
set_arm("A_age_corrected", np.ones(len(DF_ALL), bool), resid=True)


### A · 20 real splits


In [ ]:
# ======================================================================
# Cell 5 — 20 real splits: classification, geometry, reconstruction, KL
# ======================================================================
BASE_FPR = np.linspace(0, 1, 101)
BASE_REC = np.linspace(0, 1, 101)

run_rows, roc_rows, pr_rows = [], [], []
oof_recon, oof_unc, oof_pred = [], [], []
cm_sum = {"pls": np.zeros((2, 2)), "cvae_true": np.zeros((2, 2)), "cvae_neutral": np.zeros((2, 2))}
vip_real = np.zeros((N_REAL_RUNS, n_genes))
ig_real  = np.zeros((N_REAL_RUNS, n_genes))
representative = None

for r in range(N_REAL_RUNS):
    S = train_one_split(r, X_raw, y_raw)
    plsda, model = S["plsda"], S["model"]
    X_tr, X_te, y_tr, y_te = S["X_tr"], S["X_te"], S["y_tr"], S["y_te"]

    # ---- PLS-DA classification ---------------------------------------
    y_ad = (y_te == AD_IDX).astype(int)          # 1 = AD, the positive class
    Ztr_p, Zte_p = plsda.transform(X_tr), plsda.transform(X_te)
    clf_p = LogisticRegression(max_iter=1000).fit(Ztr_p, y_tr)
    prob_p = clf_p.predict_proba(Zte_p)[:, list(clf_p.classes_).index(AD_IDX)]
    pred_p = clf_p.predict(Zte_p)

    # ---- CVAE classification, both conditioning arms ------------------
    res = {}
    for mode in ("true", "neutral"):
        mu_tr, _        = encode_all(model, X_tr, y_tr, num_classes, device, mode)
        mu_te, lv_te    = encode_all(model, X_te, y_te, num_classes, device, mode)
        clf_c = LogisticRegression(max_iter=1000).fit(mu_tr, y_tr)
        res[mode] = dict(mu_tr=mu_tr, mu_te=mu_te, lv_te=lv_te,
                         prob=clf_c.predict_proba(mu_te)[:, list(clf_c.classes_).index(AD_IDX)],
                         pred=clf_c.predict(mu_te))
        with torch.no_grad():
            logits = model.classifier(torch.tensor(mu_te, dtype=torch.float32, device=device))
            res[mode]["head_prob"] = torch.softmax(logits, 1)[:, AD_IDX].cpu().numpy()
            res[mode]["head_pred"] = torch.argmax(logits, 1).cpu().numpy()

    # ---- V6: age-only reference classifier ---------------------------
    clf_a = LogisticRegression(max_iter=1000).fit(ages[S["idx_tr"]].reshape(-1, 1), y_tr)
    prob_a = clf_a.predict_proba(ages[S["idx_te"]].reshape(-1, 1))[:, list(clf_a.classes_).index(AD_IDX)]

    # ---- V7: cohort-only reference classifier --------------------------
    coh_levels = sorted(np.unique(cohorts))
    C_tr = (cohorts[S["idx_tr"]][:, None] == np.array(coh_levels)[None, :]).astype(float)
    C_te = (cohorts[S["idx_te"]][:, None] == np.array(coh_levels)[None, :]).astype(float)
    clf_k = LogisticRegression(max_iter=1000).fit(C_tr, y_tr)
    prob_k = clf_k.predict_proba(C_te)[:, list(clf_k.classes_).index(AD_IDX)]

    # ---- per-run metric row ------------------------------------------
    row = {"run": r + 1, "seed": r, "n_test": len(y_te),
           "n_test_donors": len(np.unique(donor_ids[S["idx_te"]])),
           "auc_age_only": roc_auc_score(y_ad, prob_a),
           "auc_cohort_only": roc_auc_score(y_ad, prob_k),
           "best_val_acc": S["best_val_acc"], "epochs_run": S["epochs_run"],
           "acc_pls": accuracy_score(y_te, pred_p), "auc_pls": roc_auc_score(y_ad, prob_p),
           "ap_pls": average_precision_score(y_ad, prob_p)}
    for mode in ("true", "neutral"):
        row[f"acc_cvae_lat_{mode}"]  = accuracy_score(y_te, res[mode]["pred"])
        row[f"auc_cvae_lat_{mode}"]  = roc_auc_score(y_ad, res[mode]["prob"])
        row[f"ap_cvae_lat_{mode}"]   = average_precision_score(y_ad, res[mode]["prob"])
        row[f"acc_cvae_head_{mode}"] = accuracy_score(y_te, res[mode]["head_pred"])
        row[f"auc_cvae_head_{mode}"] = roc_auc_score(y_ad, res[mode]["head_prob"])

    # ---- V6: donor-level AUC (mean P(AD) over each held-out donor's arrays)
    d_te_ids = donor_ids[S["idx_te"]]
    d_y = pd.Series(y_ad).groupby(d_te_ids).first()
    for tag, prob in (("pls", prob_p), ("cvae_lat_true", res["true"]["prob"]),
                      ("cvae_lat_neutral", res["neutral"]["prob"])):
        d_p = pd.Series(prob).groupby(d_te_ids).mean().loc[d_y.index]
        row[f"auc_{tag}_donor"] = roc_auc_score(d_y.values, d_p.values)

    # ---- latent geometry, V6: held-out arrays only ---------------------
    y_full = y_te
    geo_sets = {"pls": Zte_p,
                "cvae_true":    res["true"]["mu_te"],
                "cvae_neutral": res["neutral"]["mu_te"]}
    for name, Z in geo_sets.items():
        m = cluster_metrics(Z, y_full)
        row[f"sil_{name}"] = m["silhouette"];        row[f"ari_{name}"] = m["ari"]
        row[f"db_{name}"]  = m["davies_bouldin"];    row[f"ch_{name}"]  = m["calinski_harabasz"]
        row[f"iir_{name}"] = inter_intra_ratio(Z, y_full)

    # ---- reconstruction, held-out subjects ---------------------------
    recon_p = Zte_p @ plsda.x_loadings_.T
    mse_p   = np.mean((X_te - recon_p) ** 2, axis=1)
    mse_c   = {mode: np.mean((X_te - reconstruct_cvae(model, X_te, y_te, num_classes, device, mode)) ** 2, axis=1)
               for mode in ("true", "neutral")}
    row["mse_pls_mean"]           = mse_p.mean()
    row["mse_cvae_true_mean"]     = mse_c["true"].mean()
    row["mse_cvae_neutral_mean"]  = mse_c["neutral"].mean()

    # ---- per-subject out-of-fold records ------------------------------
    for i, gidx in enumerate(S["idx_te"]):
        oof_recon.append({"run": r + 1, "subject_id": subject_ids[gidx],
                          "group": label_encoder.classes_[y_te[i]],
                          "mse_pls": mse_p[i],
                          "mse_cvae_true": mse_c["true"][i],
                          "mse_cvae_neutral": mse_c["neutral"][i]})
        # every classifier's posterior P(AD) for this held-out subject, so the
        # confidence landscapes are drawn from out-of-fold predictions only
        prow = {"run": r + 1, "subject_id": subject_ids[gidx],
                "group": label_encoder.classes_[y_te[i]],
                "donor": donor_ids[gidx], "region": regions[gidx], "age": ages[gidx],
                "cohort": cohorts[gidx],
                "p_pls": float(prob_p[i])}
        for mode in ("true", "neutral"):
            prow[f"p_cvae_lat_{mode}"]  = float(res[mode]["prob"][i])
            prow[f"p_cvae_head_{mode}"] = float(res[mode]["head_prob"][i])
        oof_pred.append(prow)

        rec = {"run": r + 1, "subject_id": subject_ids[gidx],
               "group": label_encoder.classes_[y_te[i]]}
        for mode in ("true", "neutral"):
            mu_i, lv_i = res[mode]["mu_te"][i], res[mode]["lv_te"][i]
            rec[f"kl_{mode}"]       = float(-0.5 * np.sum(1 + lv_i - mu_i ** 2 - np.exp(lv_i)))
            rec[f"kl_perdim_{mode}"] = rec[f"kl_{mode}"] / LATENT_DIM
            rec[f"meanvar_{mode}"]   = float(np.mean(np.exp(lv_i)))
        oof_unc.append(rec)

    # ---- curves and confusion matrices -------------------------------
    for tag, prob, pred in (("pls", prob_p, pred_p),
                            ("cvae_true", res["true"]["prob"], res["true"]["pred"]),
                            ("cvae_neutral", res["neutral"]["prob"], res["neutral"]["pred"])):
        fpr, tpr, _ = roc_curve(y_ad, prob)
        roc_rows.append({"run": r + 1, "model": tag,
                         **{f"tpr_{i}": v for i, v in enumerate(np.interp(BASE_FPR, fpr, tpr))}})
        prec, rec_, _ = precision_recall_curve(y_ad, prob)
        o = np.argsort(rec_)
        pr_rows.append({"run": r + 1, "model": tag,
                        **{f"prec_{i}": v for i, v in enumerate(np.interp(BASE_REC, rec_[o], prec[o]))}})
        # rows and columns ordered [Control, AD] so tn/fp/fn/tp read with AD positive
        cm_sum[tag] += confusion_matrix(y_te, pred, labels=[1 - AD_IDX, AD_IDX])

    # ---- attribution --------------------------------------------------
    vip_real[r] = vip_scores(plsda)
    ig_real[r]  = np.abs(ig_attributions(model, X_te, y_te, num_classes, device)).mean(axis=0)

    if r == REPRESENTATIVE_SEED:
        representative = dict(S=S, res=res, Ztr_p=Ztr_p, Zte_p=Zte_p,
                              mse_p=mse_p, mse_c=mse_c, recon_p=recon_p)

    run_rows.append(row)
    print(f"run {r+1:2d}/{N_REAL_RUNS}  acc_pls={row['acc_pls']:.3f} "
          f"auc_pls={row['auc_pls']:.3f} | acc_cvae={row['acc_cvae_lat_true']:.3f} "
          f"auc_cvae={row['auc_cvae_lat_true']:.3f} | neutral auc={row['auc_cvae_lat_neutral']:.3f} "
          f"| age-only auc={row['auc_age_only']:.3f} | cohort-only auc={row['auc_cohort_only']:.3f} "
          f"| test donors={row['n_test_donors']}")

runs = pd.DataFrame(run_rows)
runs.to_csv(f"{OUT}/run_metrics.csv", index=False)
pd.DataFrame(roc_rows).to_csv(f"{OUT}/roc_curves_raw.csv", index=False)
pd.DataFrame(pr_rows).to_csv(f"{OUT}/pr_curves_raw.csv", index=False)
pd.DataFrame(oof_recon).to_csv(f"{OUT}/oof_reconstruction.csv", index=False)
pd.DataFrame(oof_unc).to_csv(f"{OUT}/oof_uncertainty.csv", index=False)
pd.DataFrame(oof_pred).to_csv(f"{OUT}/oof_predictions.csv", index=False)
pd.DataFrame([{"model": k, "tn": v[0, 0], "fp": v[0, 1], "fn": v[1, 0], "tp": v[1, 1]}
              for k, v in cm_sum.items()]).to_csv(f"{OUT}/confusion_matrices.csv", index=False)
print("\nreal runs complete")

### A · 10 label-permuted splits


In [ ]:
# ======================================================================
# Cell 6 — 10 label-permuted null splits
# ======================================================================
vip_null = np.zeros((N_NULL_RUNS, n_genes))
ig_null  = np.zeros((N_NULL_RUNS, n_genes))

for r in range(N_NULL_RUNS):
    S = train_one_split(10_000 + r, X_raw, y_raw, permute_labels=True)
    vip_null[r] = vip_scores(S["plsda"])
    ig_null[r]  = np.abs(ig_attributions(S["model"], S["X_te"], S["y_te"],
                                         num_classes, device)).mean(axis=0)
    print(f"null run {r+1}/{N_NULL_RUNS}")

np.save(f"{OUT}/vip_real.npy", vip_real); np.save(f"{OUT}/vip_null.npy", vip_null)
np.save(f"{OUT}/ig_real.npy", ig_real);   np.save(f"{OUT}/ig_null.npy", ig_null)
print("null runs complete")

### A · gene significance


In [ ]:
# ======================================================================
# Cell 7 — gene significance, stability, direction
# ======================================================================
def summarize(method, real, null, genes, top_k, stability_threshold):
    """Rank-based test against the permuted null.

    Magnitudes cannot be compared directly: mean(VIP^2) equals 1 for any
    fitted PLS model by construction, and a network trained on permuted
    labels can still produce large IG attributions by memorising noise.
    Ranking within each run sidesteps both — a gene consistently top-ranked
    across real runs but randomly ranked across null runs is real signal.
    """
    n_real, n_g = real.shape
    real_ranks = np.array([rankdata(-real[i]) for i in range(n_real)])
    null_ranks = np.array([rankdata(-null[i]) for i in range(null.shape[0])])

    pvals = np.ones(n_g)
    for j in range(n_g):
        try:
            pvals[j] = mannwhitneyu(real_ranks[:, j], null_ranks[:, j], alternative='less')[1]
        except ValueError:
            pvals[j] = 1.0
    qvals = bh_fdr(pvals)

    flags = np.zeros_like(real, dtype=bool)
    for i in range(n_real):
        flags[i, np.argsort(real[i])[::-1][:top_k]] = True
    stability = flags.mean(axis=0)

    out = pd.DataFrame({
        "gene": genes, "mean_score": real.mean(axis=0), "std_score": real.std(axis=0),
        "p_value": pvals, "fdr_q": qvals, "stability_freq": stability,
        "significant": (qvals < ALPHA) & (stability >= stability_threshold),
    }).sort_values("mean_score", ascending=False).reset_index(drop=True)
    out.insert(0, "method", method)
    return out


pls_stats  = summarize("PLS-DA_VIP", vip_real, vip_null, gene_cols, **METHOD_PARAMS["PLS-DA_VIP"])
cvae_stats = summarize("CVAE_IG",    ig_real,  ig_null,  gene_cols, **METHOD_PARAMS["CVAE_IG"])

# ---- differential expression on standardised whole-cohort expression ----
X_z = StandardScaler().fit_transform(                      # V15: arm A describes
    age_residualise(X_raw, np.arange(len(X_raw)), y_raw) if AGE_RESID else X_raw)  # age-corrected data
ad_mask = (y_raw == AD_IDX)
logfc = X_z[ad_mask].mean(axis=0) - X_z[~ad_mask].mean(axis=0)
de_p  = np.array([stats.mannwhitneyu(X_z[ad_mask, j], X_z[~ad_mask, j],
                                     alternative='two-sided')[1] for j in range(n_genes)])
de = pd.DataFrame({"gene": gene_cols, "logFC": logfc, "DE_pvalue": de_p,
                   "DE_qvalue": bh_fdr(de_p),
                   "mean_z_AD": X_z[ad_mask].mean(axis=0),
                   "mean_z_control": X_z[~ad_mask].mean(axis=0)})
de.to_csv(f"{OUT}/gene_de_stats.csv", index=False)


def add_direction(stat_df):
    m = stat_df.merge(de, on="gene", how="left")
    m["direction"] = "none"
    m.loc[(m.DE_pvalue < 0.10) & (m.logFC > 0), "direction"] = "up"
    m.loc[(m.DE_pvalue < 0.10) & (m.logFC < 0), "direction"] = "down"
    return m


pls_stats  = add_direction(pls_stats)
cvae_stats = add_direction(cvae_stats)
pls_stats.to_csv(f"{OUT}/PLSDA_gene_significance.csv", index=False)
cvae_stats.to_csv(f"{OUT}/CVAE_gene_significance.csv", index=False)

pls_sig  = set(pls_stats.loc[pls_stats.significant, "gene"])
cvae_sig = set(cvae_stats.loc[cvae_stats.significant, "gene"])
overlap  = pls_sig & cvae_sig
jaccard  = len(overlap) / max(len(pls_sig | cvae_sig), 1)
rho, rho_p = stats.spearmanr(vip_real.mean(axis=0), ig_real.mean(axis=0))

(pls_stats[pls_stats.gene.isin(overlap)][["gene", "logFC", "DE_pvalue", "direction"]]
 .drop_duplicates("gene").to_csv(f"{OUT}/consensus_significant_genes.csv", index=False))

for name, dfx, d in [("PLSDA_upregulated", pls_stats, "up"), ("PLSDA_downregulated", pls_stats, "down"),
                     ("CVAE_upregulated", cvae_stats, "up"), ("CVAE_downregulated", cvae_stats, "down")]:
    sel = dfx[dfx.significant & (dfx.direction == d)]
    sel.to_csv(f"{OUT}/{name}_genes.csv", index=False)
    print(f"{name}: {len(sel)}")

print(f"\nPLS-DA significant: {len(pls_sig)}   CVAE significant: {len(cvae_sig)}")
print(f"overlap: {len(overlap)} -> {sorted(overlap)}   Jaccard={jaccard:.4f}")
print(f"Spearman rho between mean VIP and mean IG: {rho:.4f} (p={rho_p:.3e})")

### A · latent structure, representative split


In [ ]:
# ======================================================================
# Cell 8 — latent structure of the representative split
# ======================================================================
S   = representative["S"]
res = representative["res"]
model, plsda = S["model"], S["plsda"]
X_full_rep = np.vstack([S["X_tr"], S["X_te"]])
y_full_rep = np.concatenate([S["y_tr"], S["y_te"]])

# PLS-DA latent-to-gene map: |corr(latent score, gene)|
Z_pls_full = plsda.transform(X_full_rep)
cov = (((Z_pls_full - Z_pls_full.mean(0)).T @ (X_full_rep - X_full_rep.mean(0)))
       / (len(X_full_rep) - 1))
plsd_weights = np.abs(cov / np.outer(np.maximum(Z_pls_full.std(0, ddof=1), 1e-12),
                                     np.maximum(X_full_rep.std(0, ddof=1), 1e-12)))

snr_rows, embed_rows = [], []
for tag, W, prefix in [("PLS-DA", plsd_weights, "LV"),
                       ("CVAE_true",    cvae_latent_saliency(model, X_full_rep, y_full_rep,
                                                             num_classes, device, "true"), "Z"),
                       ("CVAE_neutral", cvae_latent_saliency(model, X_full_rep, y_full_rep,
                                                             num_classes, device, "neutral"), "Z")]:
    sig = pls_sig if tag == "PLS-DA" else cvae_sig
    mask = np.array([g in sig for g in gene_cols])
    for k in range(min(LATENT_DIM, W.shape[0])):
        w = np.abs(W[k])
        snr_rows.append({"model": tag, "latent": f"{prefix}{k+1}",
                         "signal": w[mask].mean(), "noise": w[~mask].mean(),
                         "snr": w[mask].mean() / (w[~mask].mean() + 1e-12)})
    np.save(f"{OUT}/latent_gene_weights_{tag}.npy", W[:LATENT_DIM])

pd.DataFrame(snr_rows).to_csv(f"{OUT}/latent_snr.csv", index=False)

# 2-D embeddings for the geometry figure. UMAP is optional: it is not part of
# the benchmark, only of the display, so a missing package must not fail the run.
try:
    import umap
    HAVE_UMAP = True
except Exception as e:
    print(f"umap-learn unavailable ({e}); PCA and t-SNE only")
    HAVE_UMAP = False

emb = {"PLS-DA": Z_pls_full,
       "CVAE_true":    np.vstack([res["true"]["mu_tr"],    res["true"]["mu_te"]]),
       "CVAE_neutral": np.vstack([res["neutral"]["mu_tr"], res["neutral"]["mu_te"]])}
is_test = np.zeros(len(y_full_rep), bool)
is_test[len(S["y_tr"]):] = True

coord_rows = []
for tag, Z in emb.items():
    p2 = PCA(n_components=2, random_state=42).fit_transform(Z)
    t2 = TSNE(n_components=2, random_state=42,
              perplexity=min(30, max(5, len(Z) // 4)), init="pca").fit_transform(Z)
    if HAVE_UMAP:
        u2 = umap.UMAP(n_components=2, random_state=42,
                       n_neighbors=min(15, len(Z) - 1), min_dist=0.1).fit_transform(Z)
    else:
        u2 = np.full((len(Z), 2), np.nan)
    for i in range(len(Z)):
        embed_rows.append({"model": tag, "group": label_encoder.classes_[y_full_rep[i]],
                           "split": "test" if is_test[i] else "train",
                           "pca1": p2[i, 0], "pca2": p2[i, 1],
                           "tsne1": t2[i, 0], "tsne2": t2[i, 1],
                           "umap1": u2[i, 0], "umap2": u2[i, 1]})
        coord_rows.append({"model": tag, "group": label_encoder.classes_[y_full_rep[i]],
                           "split": "test" if is_test[i] else "train",
                           **{f"z{k+1}": Z[i, k] for k in range(Z.shape[1])}})
pd.DataFrame(embed_rows).to_csv(f"{OUT}/latent_embeddings.csv", index=False)
pd.DataFrame(coord_rows).to_csv(f"{OUT}/latent_coords_representative.csv", index=False)

# representative-split reconstruction scatter (subsampled for plotting)
rng = np.random.RandomState(0)
pick = rng.choice(S["X_te"].size, size=min(60000, S["X_te"].size), replace=False)
recon_c_true = reconstruct_cvae(model, S["X_te"], S["y_te"], num_classes, device, "true")
pd.DataFrame({"observed": S["X_te"].ravel()[pick],
              "recon_pls": representative["recon_p"].ravel()[pick],
              "recon_cvae": recon_c_true.ravel()[pick]}).to_csv(
    f"{OUT}/recon_scatter_representative.csv", index=False)
print("latent structure written")

### A · summary scalars


In [ ]:
# ======================================================================
# Cell 9 — every scalar the article quotes
# ======================================================================
def paired(a, b):
    t, pt = ttest_rel(a, b)
    try:    w, pw = wilcoxon(a, b)
    except ValueError: w, pw = np.nan, np.nan
    d = np.mean(a - b) / (np.std(a - b, ddof=1) + 1e-12)
    return t, pt, w, pw, d


scal = []
def put(k, v, note=""):
    scal.append({"key": k, "value": v, "note": note})

for m in ["acc_pls", "auc_pls", "ap_pls",
          "acc_cvae_lat_true", "auc_cvae_lat_true", "ap_cvae_lat_true",
          "acc_cvae_lat_neutral", "auc_cvae_lat_neutral", "ap_cvae_lat_neutral",
          "acc_cvae_head_true", "auc_cvae_head_true",
          "acc_cvae_head_neutral", "auc_cvae_head_neutral",
          "sil_pls", "ari_pls", "db_pls", "ch_pls", "iir_pls",
          "sil_cvae_true", "ari_cvae_true", "db_cvae_true", "ch_cvae_true", "iir_cvae_true",
          "sil_cvae_neutral", "ari_cvae_neutral", "db_cvae_neutral", "ch_cvae_neutral",
          "iir_cvae_neutral", "mse_pls_mean", "mse_cvae_true_mean", "mse_cvae_neutral_mean"]:
    put(f"{m}_mean", runs[m].mean()); put(f"{m}_sd", runs[m].std())

for label, a, b in [("acc_cvae_true_vs_pls", runs.acc_cvae_lat_true, runs.acc_pls),
                    ("auc_cvae_true_vs_pls", runs.auc_cvae_lat_true, runs.auc_pls),
                    ("acc_cvae_neutral_vs_pls", runs.acc_cvae_lat_neutral, runs.acc_pls),
                    ("auc_cvae_neutral_vs_pls", runs.auc_cvae_lat_neutral, runs.auc_pls),
                    ("acc_cvae_true_vs_neutral", runs.acc_cvae_lat_true, runs.acc_cvae_lat_neutral),
                    ("auc_cvae_true_vs_neutral", runs.auc_cvae_lat_true, runs.auc_cvae_lat_neutral)]:
    t, pt, w, pw, d = paired(a.values, b.values)
    put(f"{label}_t", t); put(f"{label}_p", pt); put(f"{label}_cohen_dz", d)

# reconstruction, pooled over all out-of-fold subjects
orec = pd.DataFrame(oof_recon)
for mode in ("true", "neutral"):
    a, b = orec[f"mse_cvae_{mode}"].values, orec["mse_pls"].values
    t, pt, w, pw, d = paired(a, b)
    rb = 2 * np.mean(a > b) - 1
    put(f"recon_cvae_{mode}_vs_pls_wilcoxon_p", pw)
    put(f"recon_cvae_{mode}_vs_pls_ttest_p", pt)
    put(f"recon_cvae_{mode}_vs_pls_cohen_dz", d)
    put(f"recon_cvae_{mode}_vs_pls_rank_biserial", rb)
put("recon_n_oof_samples", len(orec))
for g in orec.group.unique():
    sub = orec[orec.group == g]
    put(f"recon_mse_pls_{g}_mean", sub.mse_pls.mean())
    put(f"recon_mse_pls_{g}_sd", sub.mse_pls.std())
    put(f"recon_mse_cvae_true_{g}_mean", sub.mse_cvae_true.mean())
    put(f"recon_mse_cvae_true_{g}_sd", sub.mse_cvae_true.std())
    put(f"recon_n_{g}", len(sub))

# within a model, does reconstruction error depend on diagnosis? A model that
# rebuilt one group better than the other would confound every downstream claim.
for col, tag in [("mse_pls", "pls"), ("mse_cvae_true", "cvaetrue"),
                 ("mse_cvae_neutral", "cvaeneutral")]:
    a = orec.loc[orec.group == "AD", col].values
    b = orec.loc[orec.group != "AD", col].values
    put(f"recon_group_{tag}_mannwhitney_p", mannwhitneyu(a, b, alternative='two-sided')[1])

# confusion-matrix operating point, summed over the 20 held-out folds
cm_df = pd.DataFrame([{"model": k, "tn": v[0, 0], "fp": v[0, 1], "fn": v[1, 0], "tp": v[1, 1]}
                      for k, v in cm_sum.items()])
for _, cr in cm_df.iterrows():
    tag = cr["model"].replace("_", "")
    tn, fp, fn, tp = cr.tn, cr.fp, cr.fn, cr.tp
    put(f"cm_{tag}_sensitivity", tp / max(tp + fn, 1))
    put(f"cm_{tag}_specificity", tn / max(tn + fp, 1))
    put(f"cm_{tag}_precision", tp / max(tp + fp, 1))
    put(f"cm_{tag}_balanced_accuracy", 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)))

# out-of-fold posterior confidence: how far apart are the two classes' P(AD)?
opred = pd.DataFrame(oof_pred)
for col, tag in [("p_pls", "pls"), ("p_cvae_lat_true", "cvaelat"),
                 ("p_cvae_head_true", "cvaehead")]:
    a = opred.loc[opred.group == "AD", col].values
    b = opred.loc[opred.group != "AD", col].values
    put(f"prob_{tag}_AD_mean", a.mean())
    put(f"prob_{tag}_control_mean", b.mean())
    put(f"prob_{tag}_gap", a.mean() - b.mean())
    put(f"prob_{tag}_mannwhitney_p", mannwhitneyu(a, b, alternative='two-sided')[1])
    # fraction of subjects the model places within 0.1 of the decision threshold
    put(f"prob_{tag}_frac_ambiguous",
        float(np.mean(np.abs(opred[col].values - 0.5) < 0.1)))

# out-of-fold uncertainty: AD vs control, both conditioning arms
ounc = pd.DataFrame(oof_unc)
for mode in ("true", "neutral"):
    for col in (f"kl_{mode}", f"meanvar_{mode}"):
        ad  = ounc.loc[ounc.group == "AD", col].values
        ctl = ounc.loc[ounc.group != "AD", col].values
        u, p = mannwhitneyu(ad, ctl, alternative='two-sided')
        put(f"{col}_AD_mean", ad.mean());   put(f"{col}_AD_median", np.median(ad))
        put(f"{col}_control_mean", ctl.mean()); put(f"{col}_control_median", np.median(ctl))
        put(f"{col}_mannwhitney_p", p)
        put(f"{col}_pct_increase_AD", 100 * (ad.mean() - ctl.mean()) / abs(ctl.mean()))
put("unc_n_AD", int((ounc.group == "AD").sum()))
put("unc_n_control", int((ounc.group != "AD").sum()))

put("n_sig_plsda", len(pls_sig)); put("n_sig_cvae", len(cvae_sig))
put("n_overlap", len(overlap));   put("jaccard", jaccard)
put("consensus_genes", "|".join(sorted(overlap)))
put("spearman_rho_vip_ig", rho); put("spearman_p_vip_ig", rho_p)
put("n_total_discovered", len(pls_sig | cvae_sig))
put("n_plsda_only", len(pls_sig - cvae_sig)); put("n_cvae_only", len(cvae_sig - pls_sig))
put("n_not_significant", n_genes - len(pls_sig | cvae_sig))
put("mean_stability_plsda", pls_stats.loc[pls_stats.significant, "stability_freq"].mean())
put("mean_stability_cvae", cvae_stats.loc[cvae_stats.significant, "stability_freq"].mean())
put("n_genes", n_genes); put("n_subjects", len(X_raw))

# ---- V6 additions -------------------------------------------------------
put("n_AD", int((y_raw == AD_IDX).sum()))
put("n_donors", len(np.unique(donor_ids)))
put("n_donors_AD", len(np.unique(donor_ids[y_raw == AD_IDX])))
put("n_donors_control", len(np.unique(donor_ids[y_raw != AD_IDX])))
put("age_min", AGE_MIN)
put("mean_test_donors", runs.n_test_donors.mean())
put("n_cohorts", len(np.unique(cohorts)))
for m in ["auc_age_only", "auc_cohort_only", "auc_pls_donor", "auc_cvae_lat_true_donor",
          "auc_cvae_lat_neutral_donor"]:
    put(f"{m}_mean", runs[m].mean()); put(f"{m}_sd", runs[m].std())
# pooled out-of-fold AUC within each cohort
for coh in sorted(opred.cohort.unique()):
    sub = opred[opred.cohort == coh]
    yy = (sub.group == "AD").astype(int).values
    put(f"cohort_{coh.lower()}_n", len(sub))
    if yy.min() != yy.max():
        put(f"cohort_{coh.lower()}_auc_pls", roc_auc_score(yy, sub.p_pls))
        put(f"cohort_{coh.lower()}_auc_cvae", roc_auc_score(yy, sub.p_cvae_lat_true))
# pooled out-of-fold AUC within each brain region
for reg in sorted(opred.region.unique()):
    sub = opred[opred.region == reg]
    rtag = reg.replace(" ", "_")
    yy = (sub.group == "AD").astype(int).values
    put(f"region_{rtag}_n", len(sub))
    if yy.min() != yy.max():
        put(f"region_{rtag}_auc_pls", roc_auc_score(yy, sub.p_pls))
        put(f"region_{rtag}_auc_cvae", roc_auc_score(yy, sub.p_cvae_lat_true))
put("mean_epochs_run", runs.epochs_run.mean())

pd.DataFrame(scal).to_csv(f"{OUT}/summary_scalars.csv", index=False)
print(pd.DataFrame(scal).to_string(max_rows=400))
print("\nAll CSVs written to", OUT)
print(sorted(os.listdir(OUT)))

### A · functional enrichment


In [ ]:
# ======================================================================
# Cell 10 — functional enrichment (g:Profiler), same background for all sets
# ======================================================================
import requests

GPROFILER = "https://biit.cs.ut.ee/gprofiler/api/gost/profile/"
SETS = {
    "plsda_up":   list(pd.read_csv(f"{OUT}/PLSDA_upregulated_genes.csv").gene),
    "plsda_down": list(pd.read_csv(f"{OUT}/PLSDA_downregulated_genes.csv").gene),
    "cvae_up":    list(pd.read_csv(f"{OUT}/CVAE_upregulated_genes.csv").gene),
    "cvae_down":  list(pd.read_csv(f"{OUT}/CVAE_downregulated_genes.csv").gene),
}

os.makedirs(f"{OUT}/enrichment", exist_ok=True)
enr_counts = []
for name, genes in SETS.items():
    EMPTY = pd.DataFrame(columns=["source", "term_name", "term_id", "adjusted_p_value",
                                  "term_size", "query_size", "intersection_size",
                                  "intersections", "negative_log10_of_adjusted_p_value"])
    if not genes:
        print(f"{name}: empty gene set, skipped")
        EMPTY.to_csv(f"{OUT}/enrichment/{name}.csv", index=False)   # V6: keep the file
        enr_counts.append({"set": name, "n_genes": 0, "n_terms": 0})
        continue
    # the full preprocessed gene universe is the background, so enrichment is
    # not inflated by g:Profiler's default whole-genome domain
    payload = {"organism": "hsapiens", "query": genes, "sources": [],
               "user_threshold": 0.05, "significance_threshold_method": "fdr",
               "background": gene_cols, "domain_scope": "custom",
               "no_evidences": False}
    try:
        r = requests.post(GPROFILER, json=payload,
                          headers={"User-Agent": "ad-latent-benchmark"}, timeout=180)
        r.raise_for_status()
        body = r.json()
        res = pd.DataFrame(body.get("result", []))
    except Exception as e:                        # noqa: BLE001
        print(f"{name}: g:Profiler request failed ({type(e).__name__}: {e})")
        enr_counts.append({"set": name, "n_genes": len(genes), "n_terms": -1})
        continue

    if len(res):
        # `intersections` is one entry per *converted* query gene, holding that
        # gene's evidence codes and empty when the gene is not in the term --
        # it is not a gene list. Recover symbols by position against the query
        # order g:Profiler reports back in the metadata.
        try:
            qmeta = body["meta"]["genes_metadata"]["query"]
            order = qmeta[next(iter(qmeta))]["ensgs"]
            mapping = qmeta[next(iter(qmeta))].get("mapping", {})
            ensg2sym = {e: s for s, es in mapping.items()
                        for e in (es if isinstance(es, list) else [es])}
            qsyms = [ensg2sym.get(e, e) for e in order]
        except Exception:                          # noqa: BLE001
            qsyms = genes

        def hit_genes(v):
            if not isinstance(v, list):
                return ""
            return ",".join(qsyms[i] for i, ev in enumerate(v)
                            if ev and i < len(qsyms))

        res["hit_genes"] = res.intersections.apply(hit_genes) if "intersections" in res else ""
        keep = [c for c in ["source", "name", "native", "p_value", "term_size",
                            "query_size", "intersection_size", "hit_genes"] if c in res]
        res = res[keep].rename(columns={"name": "term_name", "native": "term_id",
                                        "p_value": "adjusted_p_value",
                                        "hit_genes": "intersections"})
        res["negative_log10_of_adjusted_p_value"] = -np.log10(res.adjusted_p_value)
        res = res.sort_values("adjusted_p_value")
    res.to_csv(f"{OUT}/enrichment/{name}.csv", index=False)
    enr_counts.append({"set": name, "n_genes": len(genes), "n_terms": len(res)})
    print(f"{name}: {len(genes)} genes -> {len(res)} enriched terms")

pd.DataFrame(enr_counts).to_csv(f"{OUT}/enrichment_counts.csv", index=False)
print("\nenrichment written to", f"{OUT}/enrichment")

## Part B — age-matched donors (sensitivity)

Donors frequency-matched on age within each cohort (5-year bands, cohort AD:control ratio kept); the unchanged pipeline runs on them.


In [ ]:
set_arm("B_age_matched", age_match_mask(seed=0), resid=False)


### B · 20 real splits


In [ ]:
# ======================================================================
# Cell 5 — 20 real splits: classification, geometry, reconstruction, KL
# ======================================================================
BASE_FPR = np.linspace(0, 1, 101)
BASE_REC = np.linspace(0, 1, 101)

run_rows, roc_rows, pr_rows = [], [], []
oof_recon, oof_unc, oof_pred = [], [], []
cm_sum = {"pls": np.zeros((2, 2)), "cvae_true": np.zeros((2, 2)), "cvae_neutral": np.zeros((2, 2))}
vip_real = np.zeros((N_REAL_RUNS, n_genes))
ig_real  = np.zeros((N_REAL_RUNS, n_genes))
representative = None

for r in range(N_REAL_RUNS):
    S = train_one_split(r, X_raw, y_raw)
    plsda, model = S["plsda"], S["model"]
    X_tr, X_te, y_tr, y_te = S["X_tr"], S["X_te"], S["y_tr"], S["y_te"]

    # ---- PLS-DA classification ---------------------------------------
    y_ad = (y_te == AD_IDX).astype(int)          # 1 = AD, the positive class
    Ztr_p, Zte_p = plsda.transform(X_tr), plsda.transform(X_te)
    clf_p = LogisticRegression(max_iter=1000).fit(Ztr_p, y_tr)
    prob_p = clf_p.predict_proba(Zte_p)[:, list(clf_p.classes_).index(AD_IDX)]
    pred_p = clf_p.predict(Zte_p)

    # ---- CVAE classification, both conditioning arms ------------------
    res = {}
    for mode in ("true", "neutral"):
        mu_tr, _        = encode_all(model, X_tr, y_tr, num_classes, device, mode)
        mu_te, lv_te    = encode_all(model, X_te, y_te, num_classes, device, mode)
        clf_c = LogisticRegression(max_iter=1000).fit(mu_tr, y_tr)
        res[mode] = dict(mu_tr=mu_tr, mu_te=mu_te, lv_te=lv_te,
                         prob=clf_c.predict_proba(mu_te)[:, list(clf_c.classes_).index(AD_IDX)],
                         pred=clf_c.predict(mu_te))
        with torch.no_grad():
            logits = model.classifier(torch.tensor(mu_te, dtype=torch.float32, device=device))
            res[mode]["head_prob"] = torch.softmax(logits, 1)[:, AD_IDX].cpu().numpy()
            res[mode]["head_pred"] = torch.argmax(logits, 1).cpu().numpy()

    # ---- V6: age-only reference classifier ---------------------------
    clf_a = LogisticRegression(max_iter=1000).fit(ages[S["idx_tr"]].reshape(-1, 1), y_tr)
    prob_a = clf_a.predict_proba(ages[S["idx_te"]].reshape(-1, 1))[:, list(clf_a.classes_).index(AD_IDX)]

    # ---- V7: cohort-only reference classifier --------------------------
    coh_levels = sorted(np.unique(cohorts))
    C_tr = (cohorts[S["idx_tr"]][:, None] == np.array(coh_levels)[None, :]).astype(float)
    C_te = (cohorts[S["idx_te"]][:, None] == np.array(coh_levels)[None, :]).astype(float)
    clf_k = LogisticRegression(max_iter=1000).fit(C_tr, y_tr)
    prob_k = clf_k.predict_proba(C_te)[:, list(clf_k.classes_).index(AD_IDX)]

    # ---- per-run metric row ------------------------------------------
    row = {"run": r + 1, "seed": r, "n_test": len(y_te),
           "n_test_donors": len(np.unique(donor_ids[S["idx_te"]])),
           "auc_age_only": roc_auc_score(y_ad, prob_a),
           "auc_cohort_only": roc_auc_score(y_ad, prob_k),
           "best_val_acc": S["best_val_acc"], "epochs_run": S["epochs_run"],
           "acc_pls": accuracy_score(y_te, pred_p), "auc_pls": roc_auc_score(y_ad, prob_p),
           "ap_pls": average_precision_score(y_ad, prob_p)}
    for mode in ("true", "neutral"):
        row[f"acc_cvae_lat_{mode}"]  = accuracy_score(y_te, res[mode]["pred"])
        row[f"auc_cvae_lat_{mode}"]  = roc_auc_score(y_ad, res[mode]["prob"])
        row[f"ap_cvae_lat_{mode}"]   = average_precision_score(y_ad, res[mode]["prob"])
        row[f"acc_cvae_head_{mode}"] = accuracy_score(y_te, res[mode]["head_pred"])
        row[f"auc_cvae_head_{mode}"] = roc_auc_score(y_ad, res[mode]["head_prob"])

    # ---- V6: donor-level AUC (mean P(AD) over each held-out donor's arrays)
    d_te_ids = donor_ids[S["idx_te"]]
    d_y = pd.Series(y_ad).groupby(d_te_ids).first()
    for tag, prob in (("pls", prob_p), ("cvae_lat_true", res["true"]["prob"]),
                      ("cvae_lat_neutral", res["neutral"]["prob"])):
        d_p = pd.Series(prob).groupby(d_te_ids).mean().loc[d_y.index]
        row[f"auc_{tag}_donor"] = roc_auc_score(d_y.values, d_p.values)

    # ---- latent geometry, V6: held-out arrays only ---------------------
    y_full = y_te
    geo_sets = {"pls": Zte_p,
                "cvae_true":    res["true"]["mu_te"],
                "cvae_neutral": res["neutral"]["mu_te"]}
    for name, Z in geo_sets.items():
        m = cluster_metrics(Z, y_full)
        row[f"sil_{name}"] = m["silhouette"];        row[f"ari_{name}"] = m["ari"]
        row[f"db_{name}"]  = m["davies_bouldin"];    row[f"ch_{name}"]  = m["calinski_harabasz"]
        row[f"iir_{name}"] = inter_intra_ratio(Z, y_full)

    # ---- reconstruction, held-out subjects ---------------------------
    recon_p = Zte_p @ plsda.x_loadings_.T
    mse_p   = np.mean((X_te - recon_p) ** 2, axis=1)
    mse_c   = {mode: np.mean((X_te - reconstruct_cvae(model, X_te, y_te, num_classes, device, mode)) ** 2, axis=1)
               for mode in ("true", "neutral")}
    row["mse_pls_mean"]           = mse_p.mean()
    row["mse_cvae_true_mean"]     = mse_c["true"].mean()
    row["mse_cvae_neutral_mean"]  = mse_c["neutral"].mean()

    # ---- per-subject out-of-fold records ------------------------------
    for i, gidx in enumerate(S["idx_te"]):
        oof_recon.append({"run": r + 1, "subject_id": subject_ids[gidx],
                          "group": label_encoder.classes_[y_te[i]],
                          "mse_pls": mse_p[i],
                          "mse_cvae_true": mse_c["true"][i],
                          "mse_cvae_neutral": mse_c["neutral"][i]})
        # every classifier's posterior P(AD) for this held-out subject, so the
        # confidence landscapes are drawn from out-of-fold predictions only
        prow = {"run": r + 1, "subject_id": subject_ids[gidx],
                "group": label_encoder.classes_[y_te[i]],
                "donor": donor_ids[gidx], "region": regions[gidx], "age": ages[gidx],
                "cohort": cohorts[gidx],
                "p_pls": float(prob_p[i])}
        for mode in ("true", "neutral"):
            prow[f"p_cvae_lat_{mode}"]  = float(res[mode]["prob"][i])
            prow[f"p_cvae_head_{mode}"] = float(res[mode]["head_prob"][i])
        oof_pred.append(prow)

        rec = {"run": r + 1, "subject_id": subject_ids[gidx],
               "group": label_encoder.classes_[y_te[i]]}
        for mode in ("true", "neutral"):
            mu_i, lv_i = res[mode]["mu_te"][i], res[mode]["lv_te"][i]
            rec[f"kl_{mode}"]       = float(-0.5 * np.sum(1 + lv_i - mu_i ** 2 - np.exp(lv_i)))
            rec[f"kl_perdim_{mode}"] = rec[f"kl_{mode}"] / LATENT_DIM
            rec[f"meanvar_{mode}"]   = float(np.mean(np.exp(lv_i)))
        oof_unc.append(rec)

    # ---- curves and confusion matrices -------------------------------
    for tag, prob, pred in (("pls", prob_p, pred_p),
                            ("cvae_true", res["true"]["prob"], res["true"]["pred"]),
                            ("cvae_neutral", res["neutral"]["prob"], res["neutral"]["pred"])):
        fpr, tpr, _ = roc_curve(y_ad, prob)
        roc_rows.append({"run": r + 1, "model": tag,
                         **{f"tpr_{i}": v for i, v in enumerate(np.interp(BASE_FPR, fpr, tpr))}})
        prec, rec_, _ = precision_recall_curve(y_ad, prob)
        o = np.argsort(rec_)
        pr_rows.append({"run": r + 1, "model": tag,
                        **{f"prec_{i}": v for i, v in enumerate(np.interp(BASE_REC, rec_[o], prec[o]))}})
        # rows and columns ordered [Control, AD] so tn/fp/fn/tp read with AD positive
        cm_sum[tag] += confusion_matrix(y_te, pred, labels=[1 - AD_IDX, AD_IDX])

    # ---- attribution --------------------------------------------------
    vip_real[r] = vip_scores(plsda)
    ig_real[r]  = np.abs(ig_attributions(model, X_te, y_te, num_classes, device)).mean(axis=0)

    if r == REPRESENTATIVE_SEED:
        representative = dict(S=S, res=res, Ztr_p=Ztr_p, Zte_p=Zte_p,
                              mse_p=mse_p, mse_c=mse_c, recon_p=recon_p)

    run_rows.append(row)
    print(f"run {r+1:2d}/{N_REAL_RUNS}  acc_pls={row['acc_pls']:.3f} "
          f"auc_pls={row['auc_pls']:.3f} | acc_cvae={row['acc_cvae_lat_true']:.3f} "
          f"auc_cvae={row['auc_cvae_lat_true']:.3f} | neutral auc={row['auc_cvae_lat_neutral']:.3f} "
          f"| age-only auc={row['auc_age_only']:.3f} | cohort-only auc={row['auc_cohort_only']:.3f} "
          f"| test donors={row['n_test_donors']}")

runs = pd.DataFrame(run_rows)
runs.to_csv(f"{OUT}/run_metrics.csv", index=False)
pd.DataFrame(roc_rows).to_csv(f"{OUT}/roc_curves_raw.csv", index=False)
pd.DataFrame(pr_rows).to_csv(f"{OUT}/pr_curves_raw.csv", index=False)
pd.DataFrame(oof_recon).to_csv(f"{OUT}/oof_reconstruction.csv", index=False)
pd.DataFrame(oof_unc).to_csv(f"{OUT}/oof_uncertainty.csv", index=False)
pd.DataFrame(oof_pred).to_csv(f"{OUT}/oof_predictions.csv", index=False)
pd.DataFrame([{"model": k, "tn": v[0, 0], "fp": v[0, 1], "fn": v[1, 0], "tp": v[1, 1]}
              for k, v in cm_sum.items()]).to_csv(f"{OUT}/confusion_matrices.csv", index=False)
print("\nreal runs complete")

### B · 10 label-permuted splits


In [ ]:
# ======================================================================
# Cell 6 — 10 label-permuted null splits
# ======================================================================
vip_null = np.zeros((N_NULL_RUNS, n_genes))
ig_null  = np.zeros((N_NULL_RUNS, n_genes))

for r in range(N_NULL_RUNS):
    S = train_one_split(10_000 + r, X_raw, y_raw, permute_labels=True)
    vip_null[r] = vip_scores(S["plsda"])
    ig_null[r]  = np.abs(ig_attributions(S["model"], S["X_te"], S["y_te"],
                                         num_classes, device)).mean(axis=0)
    print(f"null run {r+1}/{N_NULL_RUNS}")

np.save(f"{OUT}/vip_real.npy", vip_real); np.save(f"{OUT}/vip_null.npy", vip_null)
np.save(f"{OUT}/ig_real.npy", ig_real);   np.save(f"{OUT}/ig_null.npy", ig_null)
print("null runs complete")

### B · gene significance


In [ ]:
# ======================================================================
# Cell 7 — gene significance, stability, direction
# ======================================================================
def summarize(method, real, null, genes, top_k, stability_threshold):
    """Rank-based test against the permuted null.

    Magnitudes cannot be compared directly: mean(VIP^2) equals 1 for any
    fitted PLS model by construction, and a network trained on permuted
    labels can still produce large IG attributions by memorising noise.
    Ranking within each run sidesteps both — a gene consistently top-ranked
    across real runs but randomly ranked across null runs is real signal.
    """
    n_real, n_g = real.shape
    real_ranks = np.array([rankdata(-real[i]) for i in range(n_real)])
    null_ranks = np.array([rankdata(-null[i]) for i in range(null.shape[0])])

    pvals = np.ones(n_g)
    for j in range(n_g):
        try:
            pvals[j] = mannwhitneyu(real_ranks[:, j], null_ranks[:, j], alternative='less')[1]
        except ValueError:
            pvals[j] = 1.0
    qvals = bh_fdr(pvals)

    flags = np.zeros_like(real, dtype=bool)
    for i in range(n_real):
        flags[i, np.argsort(real[i])[::-1][:top_k]] = True
    stability = flags.mean(axis=0)

    out = pd.DataFrame({
        "gene": genes, "mean_score": real.mean(axis=0), "std_score": real.std(axis=0),
        "p_value": pvals, "fdr_q": qvals, "stability_freq": stability,
        "significant": (qvals < ALPHA) & (stability >= stability_threshold),
    }).sort_values("mean_score", ascending=False).reset_index(drop=True)
    out.insert(0, "method", method)
    return out


pls_stats  = summarize("PLS-DA_VIP", vip_real, vip_null, gene_cols, **METHOD_PARAMS["PLS-DA_VIP"])
cvae_stats = summarize("CVAE_IG",    ig_real,  ig_null,  gene_cols, **METHOD_PARAMS["CVAE_IG"])

# ---- differential expression on standardised whole-cohort expression ----
X_z = StandardScaler().fit_transform(                      # V15: arm A describes
    age_residualise(X_raw, np.arange(len(X_raw)), y_raw) if AGE_RESID else X_raw)  # age-corrected data
ad_mask = (y_raw == AD_IDX)
logfc = X_z[ad_mask].mean(axis=0) - X_z[~ad_mask].mean(axis=0)
de_p  = np.array([stats.mannwhitneyu(X_z[ad_mask, j], X_z[~ad_mask, j],
                                     alternative='two-sided')[1] for j in range(n_genes)])
de = pd.DataFrame({"gene": gene_cols, "logFC": logfc, "DE_pvalue": de_p,
                   "DE_qvalue": bh_fdr(de_p),
                   "mean_z_AD": X_z[ad_mask].mean(axis=0),
                   "mean_z_control": X_z[~ad_mask].mean(axis=0)})
de.to_csv(f"{OUT}/gene_de_stats.csv", index=False)


def add_direction(stat_df):
    m = stat_df.merge(de, on="gene", how="left")
    m["direction"] = "none"
    m.loc[(m.DE_pvalue < 0.10) & (m.logFC > 0), "direction"] = "up"
    m.loc[(m.DE_pvalue < 0.10) & (m.logFC < 0), "direction"] = "down"
    return m


pls_stats  = add_direction(pls_stats)
cvae_stats = add_direction(cvae_stats)
pls_stats.to_csv(f"{OUT}/PLSDA_gene_significance.csv", index=False)
cvae_stats.to_csv(f"{OUT}/CVAE_gene_significance.csv", index=False)

pls_sig  = set(pls_stats.loc[pls_stats.significant, "gene"])
cvae_sig = set(cvae_stats.loc[cvae_stats.significant, "gene"])
overlap  = pls_sig & cvae_sig
jaccard  = len(overlap) / max(len(pls_sig | cvae_sig), 1)
rho, rho_p = stats.spearmanr(vip_real.mean(axis=0), ig_real.mean(axis=0))

(pls_stats[pls_stats.gene.isin(overlap)][["gene", "logFC", "DE_pvalue", "direction"]]
 .drop_duplicates("gene").to_csv(f"{OUT}/consensus_significant_genes.csv", index=False))

for name, dfx, d in [("PLSDA_upregulated", pls_stats, "up"), ("PLSDA_downregulated", pls_stats, "down"),
                     ("CVAE_upregulated", cvae_stats, "up"), ("CVAE_downregulated", cvae_stats, "down")]:
    sel = dfx[dfx.significant & (dfx.direction == d)]
    sel.to_csv(f"{OUT}/{name}_genes.csv", index=False)
    print(f"{name}: {len(sel)}")

print(f"\nPLS-DA significant: {len(pls_sig)}   CVAE significant: {len(cvae_sig)}")
print(f"overlap: {len(overlap)} -> {sorted(overlap)}   Jaccard={jaccard:.4f}")
print(f"Spearman rho between mean VIP and mean IG: {rho:.4f} (p={rho_p:.3e})")

### B · latent structure, representative split


In [ ]:
# ======================================================================
# Cell 8 — latent structure of the representative split
# ======================================================================
S   = representative["S"]
res = representative["res"]
model, plsda = S["model"], S["plsda"]
X_full_rep = np.vstack([S["X_tr"], S["X_te"]])
y_full_rep = np.concatenate([S["y_tr"], S["y_te"]])

# PLS-DA latent-to-gene map: |corr(latent score, gene)|
Z_pls_full = plsda.transform(X_full_rep)
cov = (((Z_pls_full - Z_pls_full.mean(0)).T @ (X_full_rep - X_full_rep.mean(0)))
       / (len(X_full_rep) - 1))
plsd_weights = np.abs(cov / np.outer(np.maximum(Z_pls_full.std(0, ddof=1), 1e-12),
                                     np.maximum(X_full_rep.std(0, ddof=1), 1e-12)))

snr_rows, embed_rows = [], []
for tag, W, prefix in [("PLS-DA", plsd_weights, "LV"),
                       ("CVAE_true",    cvae_latent_saliency(model, X_full_rep, y_full_rep,
                                                             num_classes, device, "true"), "Z"),
                       ("CVAE_neutral", cvae_latent_saliency(model, X_full_rep, y_full_rep,
                                                             num_classes, device, "neutral"), "Z")]:
    sig = pls_sig if tag == "PLS-DA" else cvae_sig
    mask = np.array([g in sig for g in gene_cols])
    for k in range(min(LATENT_DIM, W.shape[0])):
        w = np.abs(W[k])
        snr_rows.append({"model": tag, "latent": f"{prefix}{k+1}",
                         "signal": w[mask].mean(), "noise": w[~mask].mean(),
                         "snr": w[mask].mean() / (w[~mask].mean() + 1e-12)})
    np.save(f"{OUT}/latent_gene_weights_{tag}.npy", W[:LATENT_DIM])

pd.DataFrame(snr_rows).to_csv(f"{OUT}/latent_snr.csv", index=False)

# 2-D embeddings for the geometry figure. UMAP is optional: it is not part of
# the benchmark, only of the display, so a missing package must not fail the run.
try:
    import umap
    HAVE_UMAP = True
except Exception as e:
    print(f"umap-learn unavailable ({e}); PCA and t-SNE only")
    HAVE_UMAP = False

emb = {"PLS-DA": Z_pls_full,
       "CVAE_true":    np.vstack([res["true"]["mu_tr"],    res["true"]["mu_te"]]),
       "CVAE_neutral": np.vstack([res["neutral"]["mu_tr"], res["neutral"]["mu_te"]])}
is_test = np.zeros(len(y_full_rep), bool)
is_test[len(S["y_tr"]):] = True

coord_rows = []
for tag, Z in emb.items():
    p2 = PCA(n_components=2, random_state=42).fit_transform(Z)
    t2 = TSNE(n_components=2, random_state=42,
              perplexity=min(30, max(5, len(Z) // 4)), init="pca").fit_transform(Z)
    if HAVE_UMAP:
        u2 = umap.UMAP(n_components=2, random_state=42,
                       n_neighbors=min(15, len(Z) - 1), min_dist=0.1).fit_transform(Z)
    else:
        u2 = np.full((len(Z), 2), np.nan)
    for i in range(len(Z)):
        embed_rows.append({"model": tag, "group": label_encoder.classes_[y_full_rep[i]],
                           "split": "test" if is_test[i] else "train",
                           "pca1": p2[i, 0], "pca2": p2[i, 1],
                           "tsne1": t2[i, 0], "tsne2": t2[i, 1],
                           "umap1": u2[i, 0], "umap2": u2[i, 1]})
        coord_rows.append({"model": tag, "group": label_encoder.classes_[y_full_rep[i]],
                           "split": "test" if is_test[i] else "train",
                           **{f"z{k+1}": Z[i, k] for k in range(Z.shape[1])}})
pd.DataFrame(embed_rows).to_csv(f"{OUT}/latent_embeddings.csv", index=False)
pd.DataFrame(coord_rows).to_csv(f"{OUT}/latent_coords_representative.csv", index=False)

# representative-split reconstruction scatter (subsampled for plotting)
rng = np.random.RandomState(0)
pick = rng.choice(S["X_te"].size, size=min(60000, S["X_te"].size), replace=False)
recon_c_true = reconstruct_cvae(model, S["X_te"], S["y_te"], num_classes, device, "true")
pd.DataFrame({"observed": S["X_te"].ravel()[pick],
              "recon_pls": representative["recon_p"].ravel()[pick],
              "recon_cvae": recon_c_true.ravel()[pick]}).to_csv(
    f"{OUT}/recon_scatter_representative.csv", index=False)
print("latent structure written")

### B · summary scalars


In [ ]:
# ======================================================================
# Cell 9 — every scalar the article quotes
# ======================================================================
def paired(a, b):
    t, pt = ttest_rel(a, b)
    try:    w, pw = wilcoxon(a, b)
    except ValueError: w, pw = np.nan, np.nan
    d = np.mean(a - b) / (np.std(a - b, ddof=1) + 1e-12)
    return t, pt, w, pw, d


scal = []
def put(k, v, note=""):
    scal.append({"key": k, "value": v, "note": note})

for m in ["acc_pls", "auc_pls", "ap_pls",
          "acc_cvae_lat_true", "auc_cvae_lat_true", "ap_cvae_lat_true",
          "acc_cvae_lat_neutral", "auc_cvae_lat_neutral", "ap_cvae_lat_neutral",
          "acc_cvae_head_true", "auc_cvae_head_true",
          "acc_cvae_head_neutral", "auc_cvae_head_neutral",
          "sil_pls", "ari_pls", "db_pls", "ch_pls", "iir_pls",
          "sil_cvae_true", "ari_cvae_true", "db_cvae_true", "ch_cvae_true", "iir_cvae_true",
          "sil_cvae_neutral", "ari_cvae_neutral", "db_cvae_neutral", "ch_cvae_neutral",
          "iir_cvae_neutral", "mse_pls_mean", "mse_cvae_true_mean", "mse_cvae_neutral_mean"]:
    put(f"{m}_mean", runs[m].mean()); put(f"{m}_sd", runs[m].std())

for label, a, b in [("acc_cvae_true_vs_pls", runs.acc_cvae_lat_true, runs.acc_pls),
                    ("auc_cvae_true_vs_pls", runs.auc_cvae_lat_true, runs.auc_pls),
                    ("acc_cvae_neutral_vs_pls", runs.acc_cvae_lat_neutral, runs.acc_pls),
                    ("auc_cvae_neutral_vs_pls", runs.auc_cvae_lat_neutral, runs.auc_pls),
                    ("acc_cvae_true_vs_neutral", runs.acc_cvae_lat_true, runs.acc_cvae_lat_neutral),
                    ("auc_cvae_true_vs_neutral", runs.auc_cvae_lat_true, runs.auc_cvae_lat_neutral)]:
    t, pt, w, pw, d = paired(a.values, b.values)
    put(f"{label}_t", t); put(f"{label}_p", pt); put(f"{label}_cohen_dz", d)

# reconstruction, pooled over all out-of-fold subjects
orec = pd.DataFrame(oof_recon)
for mode in ("true", "neutral"):
    a, b = orec[f"mse_cvae_{mode}"].values, orec["mse_pls"].values
    t, pt, w, pw, d = paired(a, b)
    rb = 2 * np.mean(a > b) - 1
    put(f"recon_cvae_{mode}_vs_pls_wilcoxon_p", pw)
    put(f"recon_cvae_{mode}_vs_pls_ttest_p", pt)
    put(f"recon_cvae_{mode}_vs_pls_cohen_dz", d)
    put(f"recon_cvae_{mode}_vs_pls_rank_biserial", rb)
put("recon_n_oof_samples", len(orec))
for g in orec.group.unique():
    sub = orec[orec.group == g]
    put(f"recon_mse_pls_{g}_mean", sub.mse_pls.mean())
    put(f"recon_mse_pls_{g}_sd", sub.mse_pls.std())
    put(f"recon_mse_cvae_true_{g}_mean", sub.mse_cvae_true.mean())
    put(f"recon_mse_cvae_true_{g}_sd", sub.mse_cvae_true.std())
    put(f"recon_n_{g}", len(sub))

# within a model, does reconstruction error depend on diagnosis? A model that
# rebuilt one group better than the other would confound every downstream claim.
for col, tag in [("mse_pls", "pls"), ("mse_cvae_true", "cvaetrue"),
                 ("mse_cvae_neutral", "cvaeneutral")]:
    a = orec.loc[orec.group == "AD", col].values
    b = orec.loc[orec.group != "AD", col].values
    put(f"recon_group_{tag}_mannwhitney_p", mannwhitneyu(a, b, alternative='two-sided')[1])

# confusion-matrix operating point, summed over the 20 held-out folds
cm_df = pd.DataFrame([{"model": k, "tn": v[0, 0], "fp": v[0, 1], "fn": v[1, 0], "tp": v[1, 1]}
                      for k, v in cm_sum.items()])
for _, cr in cm_df.iterrows():
    tag = cr["model"].replace("_", "")
    tn, fp, fn, tp = cr.tn, cr.fp, cr.fn, cr.tp
    put(f"cm_{tag}_sensitivity", tp / max(tp + fn, 1))
    put(f"cm_{tag}_specificity", tn / max(tn + fp, 1))
    put(f"cm_{tag}_precision", tp / max(tp + fp, 1))
    put(f"cm_{tag}_balanced_accuracy", 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)))

# out-of-fold posterior confidence: how far apart are the two classes' P(AD)?
opred = pd.DataFrame(oof_pred)
for col, tag in [("p_pls", "pls"), ("p_cvae_lat_true", "cvaelat"),
                 ("p_cvae_head_true", "cvaehead")]:
    a = opred.loc[opred.group == "AD", col].values
    b = opred.loc[opred.group != "AD", col].values
    put(f"prob_{tag}_AD_mean", a.mean())
    put(f"prob_{tag}_control_mean", b.mean())
    put(f"prob_{tag}_gap", a.mean() - b.mean())
    put(f"prob_{tag}_mannwhitney_p", mannwhitneyu(a, b, alternative='two-sided')[1])
    # fraction of subjects the model places within 0.1 of the decision threshold
    put(f"prob_{tag}_frac_ambiguous",
        float(np.mean(np.abs(opred[col].values - 0.5) < 0.1)))

# out-of-fold uncertainty: AD vs control, both conditioning arms
ounc = pd.DataFrame(oof_unc)
for mode in ("true", "neutral"):
    for col in (f"kl_{mode}", f"meanvar_{mode}"):
        ad  = ounc.loc[ounc.group == "AD", col].values
        ctl = ounc.loc[ounc.group != "AD", col].values
        u, p = mannwhitneyu(ad, ctl, alternative='two-sided')
        put(f"{col}_AD_mean", ad.mean());   put(f"{col}_AD_median", np.median(ad))
        put(f"{col}_control_mean", ctl.mean()); put(f"{col}_control_median", np.median(ctl))
        put(f"{col}_mannwhitney_p", p)
        put(f"{col}_pct_increase_AD", 100 * (ad.mean() - ctl.mean()) / abs(ctl.mean()))
put("unc_n_AD", int((ounc.group == "AD").sum()))
put("unc_n_control", int((ounc.group != "AD").sum()))

put("n_sig_plsda", len(pls_sig)); put("n_sig_cvae", len(cvae_sig))
put("n_overlap", len(overlap));   put("jaccard", jaccard)
put("consensus_genes", "|".join(sorted(overlap)))
put("spearman_rho_vip_ig", rho); put("spearman_p_vip_ig", rho_p)
put("n_total_discovered", len(pls_sig | cvae_sig))
put("n_plsda_only", len(pls_sig - cvae_sig)); put("n_cvae_only", len(cvae_sig - pls_sig))
put("n_not_significant", n_genes - len(pls_sig | cvae_sig))
put("mean_stability_plsda", pls_stats.loc[pls_stats.significant, "stability_freq"].mean())
put("mean_stability_cvae", cvae_stats.loc[cvae_stats.significant, "stability_freq"].mean())
put("n_genes", n_genes); put("n_subjects", len(X_raw))

# ---- V6 additions -------------------------------------------------------
put("n_AD", int((y_raw == AD_IDX).sum()))
put("n_donors", len(np.unique(donor_ids)))
put("n_donors_AD", len(np.unique(donor_ids[y_raw == AD_IDX])))
put("n_donors_control", len(np.unique(donor_ids[y_raw != AD_IDX])))
put("age_min", AGE_MIN)
put("mean_test_donors", runs.n_test_donors.mean())
put("n_cohorts", len(np.unique(cohorts)))
for m in ["auc_age_only", "auc_cohort_only", "auc_pls_donor", "auc_cvae_lat_true_donor",
          "auc_cvae_lat_neutral_donor"]:
    put(f"{m}_mean", runs[m].mean()); put(f"{m}_sd", runs[m].std())
# pooled out-of-fold AUC within each cohort
for coh in sorted(opred.cohort.unique()):
    sub = opred[opred.cohort == coh]
    yy = (sub.group == "AD").astype(int).values
    put(f"cohort_{coh.lower()}_n", len(sub))
    if yy.min() != yy.max():
        put(f"cohort_{coh.lower()}_auc_pls", roc_auc_score(yy, sub.p_pls))
        put(f"cohort_{coh.lower()}_auc_cvae", roc_auc_score(yy, sub.p_cvae_lat_true))
# pooled out-of-fold AUC within each brain region
for reg in sorted(opred.region.unique()):
    sub = opred[opred.region == reg]
    rtag = reg.replace(" ", "_")
    yy = (sub.group == "AD").astype(int).values
    put(f"region_{rtag}_n", len(sub))
    if yy.min() != yy.max():
        put(f"region_{rtag}_auc_pls", roc_auc_score(yy, sub.p_pls))
        put(f"region_{rtag}_auc_cvae", roc_auc_score(yy, sub.p_cvae_lat_true))
put("mean_epochs_run", runs.epochs_run.mean())

pd.DataFrame(scal).to_csv(f"{OUT}/summary_scalars.csv", index=False)
print(pd.DataFrame(scal).to_string(max_rows=400))
print("\nAll CSVs written to", OUT)
print(sorted(os.listdir(OUT)))

### B · functional enrichment


In [ ]:
# ======================================================================
# Cell 10 — functional enrichment (g:Profiler), same background for all sets
# ======================================================================
import requests

GPROFILER = "https://biit.cs.ut.ee/gprofiler/api/gost/profile/"
SETS = {
    "plsda_up":   list(pd.read_csv(f"{OUT}/PLSDA_upregulated_genes.csv").gene),
    "plsda_down": list(pd.read_csv(f"{OUT}/PLSDA_downregulated_genes.csv").gene),
    "cvae_up":    list(pd.read_csv(f"{OUT}/CVAE_upregulated_genes.csv").gene),
    "cvae_down":  list(pd.read_csv(f"{OUT}/CVAE_downregulated_genes.csv").gene),
}

os.makedirs(f"{OUT}/enrichment", exist_ok=True)
enr_counts = []
for name, genes in SETS.items():
    EMPTY = pd.DataFrame(columns=["source", "term_name", "term_id", "adjusted_p_value",
                                  "term_size", "query_size", "intersection_size",
                                  "intersections", "negative_log10_of_adjusted_p_value"])
    if not genes:
        print(f"{name}: empty gene set, skipped")
        EMPTY.to_csv(f"{OUT}/enrichment/{name}.csv", index=False)   # V6: keep the file
        enr_counts.append({"set": name, "n_genes": 0, "n_terms": 0})
        continue
    # the full preprocessed gene universe is the background, so enrichment is
    # not inflated by g:Profiler's default whole-genome domain
    payload = {"organism": "hsapiens", "query": genes, "sources": [],
               "user_threshold": 0.05, "significance_threshold_method": "fdr",
               "background": gene_cols, "domain_scope": "custom",
               "no_evidences": False}
    try:
        r = requests.post(GPROFILER, json=payload,
                          headers={"User-Agent": "ad-latent-benchmark"}, timeout=180)
        r.raise_for_status()
        body = r.json()
        res = pd.DataFrame(body.get("result", []))
    except Exception as e:                        # noqa: BLE001
        print(f"{name}: g:Profiler request failed ({type(e).__name__}: {e})")
        enr_counts.append({"set": name, "n_genes": len(genes), "n_terms": -1})
        continue

    if len(res):
        # `intersections` is one entry per *converted* query gene, holding that
        # gene's evidence codes and empty when the gene is not in the term --
        # it is not a gene list. Recover symbols by position against the query
        # order g:Profiler reports back in the metadata.
        try:
            qmeta = body["meta"]["genes_metadata"]["query"]
            order = qmeta[next(iter(qmeta))]["ensgs"]
            mapping = qmeta[next(iter(qmeta))].get("mapping", {})
            ensg2sym = {e: s for s, es in mapping.items()
                        for e in (es if isinstance(es, list) else [es])}
            qsyms = [ensg2sym.get(e, e) for e in order]
        except Exception:                          # noqa: BLE001
            qsyms = genes

        def hit_genes(v):
            if not isinstance(v, list):
                return ""
            return ",".join(qsyms[i] for i, ev in enumerate(v)
                            if ev and i < len(qsyms))

        res["hit_genes"] = res.intersections.apply(hit_genes) if "intersections" in res else ""
        keep = [c for c in ["source", "name", "native", "p_value", "term_size",
                            "query_size", "intersection_size", "hit_genes"] if c in res]
        res = res[keep].rename(columns={"name": "term_name", "native": "term_id",
                                        "p_value": "adjusted_p_value",
                                        "hit_genes": "intersections"})
        res["negative_log10_of_adjusted_p_value"] = -np.log10(res.adjusted_p_value)
        res = res.sort_values("adjusted_p_value")
    res.to_csv(f"{OUT}/enrichment/{name}.csv", index=False)
    enr_counts.append({"set": name, "n_genes": len(genes), "n_terms": len(res)})
    print(f"{name}: {len(genes)} genes -> {len(res)} enriched terms")

pd.DataFrame(enr_counts).to_csv(f"{OUT}/enrichment_counts.csv", index=False)
print("\nenrichment written to", f"{OUT}/enrichment")

## Part C — both arms against V7


In [ ]:
# ======================================================================
# Part C — both arms against V7 (no age handling)
# ======================================================================
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

v7_hits = [h for h in glob.glob("/kaggle/input/**/summary_scalars.csv", recursive=True)]
v7_hits = [h for h in v7_hits if "v7admerged" in h] or v7_hits
assert v7_hits, "attach the v7admerged notebook output as an input"
ARMS = {"V7": os.path.dirname(v7_hits[0]),
        "A_age_corrected": f"{ROOT_OUT}/A_age_corrected",
        "B_age_matched": f"{ROOT_OUT}/B_age_matched"}
LABEL = {"V7": "V7, no age handling", "A_age_corrected": "A, age-corrected", "B_age_matched": "B, age-matched"}


def per_run_auc(d, col):
    v = [roc_auc_score(g.group == "AD", g[col]) for _, g in d.groupby("run") if g.group.nunique() == 2]
    return (np.mean(v), np.std(v)) if v else (np.nan, np.nan)


def adj_auc(d, col):
    """Covariate-adjusted AUC: the score corrected for age on that split's test
    controls, then AUC; one value per split, averaged."""
    out = []
    for _, g in d.groupby("run"):
        if g.group.nunique() < 2:
            continue
        p = np.clip(g[col].values, 1e-6, 1 - 1e-6); s = np.log(p / (1 - p))
        k = (g.group == "Control").values
        if k.sum() < 3:
            continue
        b = np.polyfit(g.age.values[k], s[k], 1)
        out.append(roc_auc_score(g.group == "AD", s - np.polyval(b, g.age.values)))
    return np.mean(out) if out else np.nan


head, coh, genes = [], [], {}
for arm, d_ in ARMS.items():
    rm = pd.read_csv(f"{d_}/run_metrics.csv")
    op = pd.read_csv(f"{d_}/oof_predictions.csv")
    P = pd.read_csv(f"{d_}/PLSDA_gene_significance.csv"); Cv = pd.read_csv(f"{d_}/CVAE_gene_significance.csv")
    genes[arm] = (set(P.gene[P.significant]), set(Cv.gene[Cv.significant]))
    cs_ = pd.read_csv(f"{d_}/cohort_summary.csv").iloc[0]
    row = {"arm": LABEL[arm], "arrays": int(cs_.n_subjects), "donors": int(cs_.n_donors)}
    for tag, col in (("pls", "auc_pls"), ("cvae_agn", "auc_cvae_lat_neutral"), ("cvae_cond", "auc_cvae_lat_true"),
                     ("age_only", "auc_age_only"), ("cohort_only", "auc_cohort_only"),
                     ("pls_donor", "auc_pls_donor"), ("cvae_agn_donor", "auc_cvae_lat_neutral_donor")):
        row[f"auc_{tag}"] = rm[col].mean(); row[f"auc_{tag}_sd"] = rm[col].std()
    t = ttest_rel(rm.auc_cvae_lat_neutral, rm.auc_pls)
    row["cvae_agn_vs_pls_p"] = t.pvalue
    row["cvae_agn_wins"] = int((rm.auc_cvae_lat_neutral > rm.auc_pls).sum())
    row["n_genes_pls"], row["n_genes_cvae"] = len(genes[arm][0]), len(genes[arm][1])
    row["n_shared"] = len(genes[arm][0] & genes[arm][1])
    head.append(row)
    for c, d in op.groupby("cohort"):
        r = {"arm": LABEL[arm], "cohort": c}
        for tag, col in (("pls", "p_pls"), ("cvae_agn", "p_cvae_lat_neutral")):
            r[f"auc_{tag}"], r[f"auc_{tag}_sd"] = per_run_auc(d, col)
            r[f"auc_{tag}_age_adjusted"] = adj_auc(d, col)
        r["auc_age_alone"] = per_run_auc(d, "age")[0]
        coh.append(r)

H = pd.DataFrame(head); K = pd.DataFrame(coh)
H.to_csv(f"{ROOT_OUT}/comparison_headline.csv", index=False)
K.to_csv(f"{ROOT_OUT}/comparison_per_cohort.csv", index=False)
print(H.round(3).T.to_string()); print(); print(K.round(3).to_string(index=False))

# ---- genes: what survives, and are they age genes? --------------------------------
p7, c7 = genes["V7"]; core7 = p7 & c7
grows = []
for arm in ("A_age_corrected", "B_age_matched"):
    pa, ca = genes[arm]
    grows += [{"arm": LABEL[arm], "list": "PLS-DA", "V7": len(p7), "this_arm": len(pa), "kept_from_V7": len(p7 & pa),
               "jaccard_with_V7": len(p7 & pa) / max(len(p7 | pa), 1)},
              {"arm": LABEL[arm], "list": "CVAE", "V7": len(c7), "this_arm": len(ca), "kept_from_V7": len(c7 & ca),
               "jaccard_with_V7": len(c7 & ca) / max(len(c7 | ca), 1)},
              {"arm": LABEL[arm], "list": "shared by both models", "V7": len(core7), "this_arm": len(pa & ca),
               "kept_from_V7": len(core7 & pa & ca),
               "jaccard_with_V7": len(core7 & pa & ca) / max(len(core7 | (pa & ca)), 1)}]
G = pd.DataFrame(grows); G.to_csv(f"{ROOT_OUT}/comparison_genes.csv", index=False)
print(G.round(3).to_string(index=False))

M = pd.DataFrame({"gene": sorted(core7)})
for arm in ARMS:
    pa, ca = genes[arm]
    M[f"{arm}_PLS-DA"] = M.gene.isin(pa); M[f"{arm}_CVAE"] = M.gene.isin(ca)
M.to_csv(f"{ROOT_OUT}/v7_shared_genes_by_arm.csv", index=False)
print(M.to_string(index=False))

# age association of genes in controls (raw merged data, Spearman per cohort-pooled z)
ctrl = (Y_ALL != AD_IDX)
from scipy.stats import spearmanr
rho = np.array([spearmanr(DF_ALL.age.values[ctrl], X_ALL[ctrl, j]).correlation for j in range(n_genes)])
pv = np.array([spearmanr(DF_ALL.age.values[ctrl], X_ALL[ctrl, j]).pvalue for j in range(n_genes)])
qv = bh_fdr(pv); age_gene = dict(zip(gene_cols, qv < 0.05))
arows = [{"arm": "all genes", "list": "background", "n": n_genes, "frac_age_genes": float(np.mean(qv < 0.05))}]
for arm in ARMS:
    for nm, s in zip(("PLS-DA", "CVAE"), genes[arm]):
        arows.append({"arm": LABEL[arm], "list": nm, "n": len(s),
                      "frac_age_genes": float(np.mean([age_gene[g] for g in s if g in age_gene])) if s else np.nan})
AG = pd.DataFrame(arows); AG.to_csv(f"{ROOT_OUT}/comparison_age_genes.csv", index=False)
print(AG.round(3).to_string(index=False))

# ---- two quick-look figures ----------------------------------------------------------
NAVY, WINE, VIOLET, GREY = "#24486b", "#8f3a4f", "#6f6396", "#8a8a92"
ACOL = {"V7, no age handling": GREY, "A, age-corrected": NAVY, "B, age-matched": VIOLET}
plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
cos = sorted(K.cohort.unique())
for ax, tag, ttl in ((axes[0], "pls", "PLS-DA"), (axes[1], "cvae_agn", "CVAE, label-agnostic")):
    for j, (arm, col) in enumerate(ACOL.items()):
        s = K[K.arm == arm].set_index("cohort").reindex(cos)
        y = np.arange(len(cos)) + (j - 1) * 0.24
        ax.errorbar(s[f"auc_{tag}"], y, xerr=s[f"auc_{tag}_sd"], fmt="o", color=col, ms=4, lw=0.8, label=arm)
        ax.scatter(s.auc_age_alone, y, marker="D", s=14, facecolor="white", edgecolor=col, zorder=3)
    ax.set_yticks(range(len(cos))); ax.set_yticklabels(cos); ax.invert_yaxis()
    ax.axvline(0.5, color=GREY, lw=0.6, ls=":"); ax.set_xlim(0.4, 1.0); ax.set_title(ttl, loc="left")
    ax.set_xlabel("test ROC-AUC within cohort (dot) | age alone (hollow diamond)")
axes[0].legend(loc="lower left", frameon=False)
fig.tight_layout(); fig.savefig(f"{ROOT_OUT}/v15_per_cohort_auc.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.6))
mat = M.set_index("gene")[[c for c in M.columns if c != "gene"]].astype(int).T
ax.imshow(mat.values, cmap="Blues", vmin=0, vmax=1.4, aspect="auto")
ax.set_yticks(range(len(mat))); ax.set_yticklabels(mat.index)
ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=90, fontstyle="italic")
ax.set_title(f"The {len(core7)} genes V7's two models shared: still selected after age is taken out?", loc="left")
fig.tight_layout(); fig.savefig(f"{ROOT_OUT}/v15_shared_genes.png", dpi=200); plt.show()

# ---- plain-text report -------------------------------------------------------------------
L = ["V15ADage — the merged benchmark with age taken out", "=" * 52, ""]
for _, r in H.iterrows():
    L.append(f"{r.arm}: {int(r.donors)} donors | AUC PLS-DA {r.auc_pls:.3f} ± {r.auc_pls_sd:.3f}, "
             f"CVAE agn. {r.auc_cvae_agn:.3f} ± {r.auc_cvae_agn_sd:.3f} (paired p {r.cvae_agn_vs_pls_p:.2g}, "
             f"CVAE higher in {r.cvae_agn_wins}/20) | age alone {r.auc_age_only:.3f} | "
             f"genes {r.n_genes_pls} / {r.n_genes_cvae}, shared {r.n_shared}")
L += ["", "Per cohort (PLS-DA / CVAE agn. / age alone):"]
for _, r in K.iterrows():
    L.append(f"  {r.arm:22s} {r.cohort:10s} {r.auc_pls:.3f} / {r.auc_cvae_agn:.3f} / {r.auc_age_alone:.3f}")
L += ["", "Genes kept from V7:"] + [f"  {r.arm:22s} {r.list:22s} {r.kept_from_V7} of {r.V7}  (this arm {r.this_arm})"
                                    for _, r in G.iterrows()]
L += ["", "Fraction of selected genes that are age-associated in controls (q<0.05):"] + \
     [f"  {r.arm:22s} {r.list:10s} {r.frac_age_genes:.3f}  (n={r.n})" for _, r in AG.iterrows()]
open(f"{ROOT_OUT}/V15_REPORT.txt", "w").write("\n".join(L) + "\n")
print("\n".join(L))
